# Recuperação de Tickers Históricos via Macrotrends / Stooq

Coleta dados dos 34 tickers pendentes no `ausencias_report.md`:
- 31 irrecuperáveis via Yahoo/Tiingo
- 3 parciais (FOX, FOXA, IR)

**Fluxo:** macrotrends (scraping) → stooq (CSV direto) → falha documentada  
**Saída:** `data_bases/external/macrotrends_recovery/{TICKER}.csv` (não sobrescreve prices/ diretamente)  
**Formato:** `Date,{TICKER}` — igual ao padrão de `data_bases/prices/`

In [18]:
import requests
import re
import json
import time
import shutil
import pandas as pd
from pathlib import Path

NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PRICES_DIR    = PROJECT_ROOT / "data_bases" / "prices"
RECOVERY_DIR  = PROJECT_ROOT / "data_bases" / "external" / "macrotrends_recovery"
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

print("Prices dir :", PRICES_DIR)
print("Recovery dir:", RECOVERY_DIR)

Prices dir : C:\Users\jvlei\Desktop\TCC-pair-trading\new_aproach\data_bases\prices
Recovery dir: C:\Users\jvlei\Desktop\TCC-pair-trading\new_aproach\data_bases\external\macrotrends_recovery


In [19]:
import re
import json
import time
import pandas as pd
from io import StringIO
from pathlib import Path

# pip install cloudscraper
import cloudscraper

SCRAPER = cloudscraper.create_scraper(
    browser={"browser": "chrome", "platform": "windows", "mobile": False}
)


def fetch_macrotrends(mt_ticker: str, slug: str, start: str, end: str) -> pd.DataFrame:
    url = f"https://www.macrotrends.net/stocks/charts/{mt_ticker}/{slug}/stock-price-history"
    try:
        r = SCRAPER.get(url, timeout=30)
        print(f"    macrotrends status: {r.status_code}")
    except Exception as e:
        print(f"    macrotrends error: {e}")
        return pd.DataFrame()

    match = re.search(r'var originalData\s*=\s*(\[.*?\]);', r.text, re.DOTALL)
    if not match:
        # Mostrar trecho do HTML para diagnóstico
        snippet = r.text[:300].replace('\n', ' ')
        print(f"    var originalData não encontrada. HTML: {snippet!r}")
        return pd.DataFrame()

    try:
        raw = json.loads(match.group(1))
    except json.JSONDecodeError as e:
        print(f"    JSON parse error: {e}")
        return pd.DataFrame()

    if not raw:
        print(f"    Array vazio")
        return pd.DataFrame()

    df = pd.DataFrame(raw)
    df.columns = [c.lower() for c in df.columns]

    close_col = next((c for c in ["close", "adjclose", "adj_close"] if c in df.columns), None)
    if close_col is None or "date" not in df.columns:
        print(f"    Colunas inesperadas: {list(df.columns)}")
        return pd.DataFrame()

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date", close_col])
    df = df[(df["date"] >= start) & (df["date"] <= end)]
    df = df[["date", close_col]].copy()
    df.columns = ["Date", "close"]
    return df.sort_values("Date").reset_index(drop=True)


# --- Teste rápido com 2 tickers antes de rodar tudo ---
print("=== TESTE MACROTRENDS + CLOUDSCRAPER ===\n")
for mt_ticker, slug in [("MON", "monsanto"), ("META", "meta-platforms")]:
    print(f"[{mt_ticker}/{slug}]")
    df = fetch_macrotrends(mt_ticker, slug, "2016-01-01", "2018-06-07")
    if not df.empty:
        print(f"  ✅ {len(df)} linhas | {df['Date'].min().date()} → {df['Date'].max().date()}")
        print(df.head(3).to_string(index=False))
    print()

=== TESTE MACROTRENDS + CLOUDSCRAPER ===

[MON/monsanto]
    macrotrends status: 403
    var originalData não encontrada. HTML: '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal'

[META/meta-platforms]
    macrotrends status: 403
    var originalData não encontrada. HTML: '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal'



In [5]:
# Formato: (orig_ticker, mt_ticker, mt_slug, start, end)
# - orig_ticker : nome do arquivo CSV de saída (ex: 'FB.csv')
# - mt_ticker   : ticker usado pela macrotrends na URL (pode ser o sucessor)
# - mt_slug     : slug da URL macrotrends
# - start/end   : janela de datas necessária

targets = [
    # ==================================================================
    # GRUPO 1 — Ticker reciclado: sucessor com ticker diferente
    # Macrotrends mantém histórico pré-rename sob o ticker atual
    # ==================================================================
    ("FB",    "META",  "meta-platforms",             "2016-01-01", "2021-10-28"),
    ("LB",    "BBWI",  "bath-body-works",            "2016-01-01", "2021-08-02"),
    ("ANTM",  "ELV",   "elevance-health",            "2016-01-01", "2022-06-28"),
    ("STI",   "TFC",   "truist-financial",           "2016-01-01", "2019-12-06"),
    ("VIAC",  "PARA",  "paramount-global",           "2019-12-05", "2022-02-15"),
    ("FBHS",  "FBIN",  "fortune-brands-innovations", "2016-01-01", "2022-11-08"),
    ("TMK",   "GL",    "globe-life",                 "2016-01-01", "2019-08-08"),
    ("DISCA", "WBD",   "warner-bros-discovery",      "2016-01-01", "2022-02-15"),
    ("PKI",   "RVTY",  "revvity",                    "2016-01-01", "2023-05-04"),
    # HCP renomeou para PEAK em 2019 — mesma empresa, página PEAK cobre histórico completo
    ("HCP",   "PEAK",  "healthpeak-properties",      "2016-01-01", "2019-06-30"),
    # ARNC original → split em HWM (Howmet) + novo ARNC em abr/2020
    # HWM é o sucessor legal do ARNC original
    ("ARNC",  "HWM",   "howmet-aerospace",           "2016-11-01", "2020-04-01"),

    # ==================================================================
    # GRUPO 2 — Empresa adquirida/dissolvida: página histórica própria
    # Macrotrends mantém páginas de empresas extintas
    # ==================================================================
    ("MON",   "MON",   "monsanto",                   "2016-01-01", "2018-06-07"),
    ("SE",    "SE",    "spectra-energy",             "2016-01-01", "2017-02-27"),
    ("TE",    "TE",    "teco-energy",                "2016-01-01", "2016-07-10"),
    ("CA",    "CA",    "ca-technologies",            "2016-01-01", "2018-11-05"),
    ("EMC",   "EMC",   "emc",                        "2016-01-01", "2016-09-10"),
    ("APC",   "APC",   "anadarko-petroleum",         "2016-01-01", "2019-08-08"),
    ("DNB",   "DNB",   "dun-bradstreet",             "2016-01-01", "2019-02-08"),
    ("DO",    "DO",    "diamond-offshore-drilling",  "2016-01-01", "2020-04-26"),
    ("FTR",   "FTR",   "frontier-communications",    "2016-01-01", "2020-04-14"),
    ("CBS",   "CBS",   "cbs",                        "2016-01-01", "2019-12-05"),
    ("ENDP",  "ENDP",  "endo-international",         "2016-01-01", "2022-08-16"),
    ("MNK",   "MNK",   "mallinckrodt",               "2016-01-01", "2020-10-12"),

    # ==================================================================
    # GRUPO 3 — Tiingo Free Tier sem dados
    # ==================================================================
    ("BLL",   "BLL",   "ball",                       "2016-01-01", "2022-04-11"),
    ("CDAY",  "CDAY",  "ceridian-hcm",               "2021-09-20", "2023-10-18"),
    ("FRC",   "FRC",   "first-republic-bank",        "2018-07-01", "2023-05-01"),
    ("GPS",   "GPS",   "gap",                        "2016-01-01", "2022-01-10"),
    ("MMC",   "MMC",   "marsh-mclennan",             "2016-01-01", "2025-12-31"),
    ("PEAK",  "PEAK",  "healthpeak-properties",      "2019-07-01", "2024-02-01"),
    ("RE",    "RE",    "everest-re-group",            "2015-07-01", "2023-06-20"),
    ("WRK",   "WRK",   "westrock",                   "2016-01-01", "2024-07-05"),

    # ==================================================================
    # GRUPO 4 — Parciais (arquivo existe mas sem período anterior)
    # ==================================================================
    # FOX/FOXA: arquivos têm Fox Corp (2019+). Queremos 21CF (2016-2019).
    # Macrotrends pode ter histórico 21CF sob o ticker FOX/FOXA.
    # Se falhar, 21CF não está disponível via macrotrends.
    ("FOX",   "FOX",   "fox-corporation",            "2016-01-01", "2019-03-18"),
    ("FOXA",  "FOXA",  "fox-corporation",            "2016-01-01", "2019-03-18"),
    # IR: arquivo tem 2017-05+. Queremos 2016-2017.
    # Old Ingersoll-Rand → split em TT (Trane) + novo IR em fev/2020
    # TT é o sucessor legal do old IR — página TT deve ter histórico completo
    ("IR",    "TT",    "trane-technologies",         "2016-01-01", "2017-05-11"),
]

print(f"Total de targets: {len(targets)}")

Total de targets: 34


In [6]:
results = []

for orig_ticker, mt_ticker, slug, start, end in targets:
    print(f"\n{'─'*55}")
    print(f"[{orig_ticker}] → {mt_ticker}/{slug}  ({start[:7]} → {end[:7]})")

    df = fetch_macrotrends(mt_ticker, slug, start, end)

    if not df.empty:
        out = df.rename(columns={"close": orig_ticker})
        out_path = RECOVERY_DIR / f"{orig_ticker}.csv"
        out.to_csv(out_path, index=False)
        print(f"    ✅ {len(out)} linhas → {out_path.name}")
        results.append({"ticker": orig_ticker, "status": "ok", "rows": len(out),
                        "start": str(df["Date"].min())[:10], "end": str(df["Date"].max())[:10]})
    else:
        print(f"    ❌ Não encontrado")
        results.append({"ticker": orig_ticker, "status": "fail", "rows": 0,
                        "start": "-", "end": "-"})

    time.sleep(3)  # respeitar rate limit


───────────────────────────────────────────────────────
[FB] → META/meta-platforms  (2016-01 → 2021-10)
    macrotrends status: 403
    var originalData não encontrada. HTML: '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal'
    ❌ Não encontrado

───────────────────────────────────────────────────────
[LB] → BBWI/bath-body-works  (2016-01 → 2021-08)
    macrotrends status: 403
    var originalData não encontrada. HTML: '<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scal'
    ❌ Não encontrado

KeyboardInterrupt: 

In [21]:
df_res = pd.DataFrame(results)
ok   = df_res[df_res.status == "ok"]
fail = df_res[df_res.status != "ok"]

print(f"\n{'='*55}")
print(f"RESUMO DA COLETA")
print(f"{'='*55}")
print(f"✅ Recuperados : {len(ok)} / {len(df_res)}")
print(f"❌ Falharam    : {len(fail)} / {len(df_res)}")

if not ok.empty:
    print("\n--- Recuperados ---")
    print(ok[["ticker", "source", "rows", "start", "end"]].to_string(index=False))

if not fail.empty:
    print("\n--- Falharam ---")
    print(fail[["ticker"]].to_string(index=False))


RESUMO DA COLETA
✅ Recuperados : 0 / 3
❌ Falharam    : 3 / 3

--- Falharam ---
ticker
    FB
    LB
  ANTM


## Incorporar dados recuperados em `prices/`

Execute a célula abaixo **depois de revisar** os arquivos em `macrotrends_recovery/`.

- **Sem arquivo existente** → copia diretamente para `prices/`
- **Arquivo parcial existente** (FOX, FOXA, IR) → faz merge, sem duplicatas, ordenado por data

In [20]:
def incorporate_recovered(orig_ticker: str, dry_run: bool = True):
    rec_file   = RECOVERY_DIR / f"{orig_ticker}.csv"
    price_file = PRICES_DIR   / f"{orig_ticker}.csv"

    if not rec_file.exists():
        print(f"[{orig_ticker}] Sem arquivo em recovery/ — pulando")
        return

    rec_df = pd.read_csv(rec_file, parse_dates=["Date"])

    if price_file.exists():
        existing = pd.read_csv(price_file, parse_dates=["Date"])
        merged = (
            pd.concat([rec_df, existing])
            .drop_duplicates("Date")
            .sort_values("Date")
            .reset_index(drop=True)
        )
        action = f"merge: {len(rec_df)} novas + {len(existing)} existentes = {len(merged)} total"
        if not dry_run:
            merged.to_csv(price_file, index=False)
    else:
        merged = rec_df
        action = f"cópia nova: {len(merged)} linhas"
        if not dry_run:
            shutil.copy(rec_file, price_file)

    status = "[DRY RUN]" if dry_run else "[GRAVADO]"
    print(f"{status} [{orig_ticker}] {action}")


# --- Incorporar apenas os recuperados (dry_run=True para preview) ---
recovered_tickers = [r["ticker"] for r in results if r["status"] == "ok"]
print(f"Tickers prontos para incorporar: {recovered_tickers}\n")

for ticker in recovered_tickers:
    incorporate_recovered(ticker, dry_run=False)   # mude para False para gravar

Tickers prontos para incorporar: []



---
## Parte B — Retry Yahoo / Tiingo para grupos não-reciclados

Tickers que **não** são reciclados — o problema foi erro da API ou limitação do free tier do Tiingo.

| Grupo | Tickers | Estratégia |
|-------|---------|------------|
| API errors | CBS, ENDP, MNK, APC | Tiingo com datas exatas de membership |
| Tiingo free tier (ativos) | BLL, CDAY, GPS, MMC, RE | Yahoo primeiro, Tiingo como fallback |
| Tiingo free tier (renomeados) | ANTM, FBHS, FRC, PEAK, PKI, TMK, WRK | Tiingo com datas exatas |
| Parcial | IR | Yahoo (ticker ainda existe) |

Saída → `macrotrends_recovery/` (mesma pasta de antes). Incorporar com a função da seção anterior.

In [10]:
import yfinance as yf
import requests
import time
import pandas as pd
from pathlib import Path

TIINGO_TOKEN = "dadfd331f2cb44969b8f7468006d20ad62b13262"
PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

# (ticker, tiingo_start, tiingo_end, yahoo_ativo)
# tiingo_start/end = janela de membership exata (evita pegar ticker reciclado)
# yahoo_ativo = True → ticker ainda existe no Yahoo com esse nome
targets_b = [
    # --- Grupo 2: API errors / falência / aquisição ---
    ("CBS",  "2015-07-01", "2019-12-05", False),   # merged → ViacomCBS dez/2019
    ("ENDP", "2015-07-01", "2022-08-16", False),   # falência ago/2022
    ("MNK",  "2015-07-01", "2020-10-12", False),   # falência out/2020
    ("APC",  "2015-07-01", "2019-08-08", False),   # adquirida OXY ago/2019
    # --- Grupo 3: ainda ativos com mesmo ticker ---
    ("BLL",  "2015-07-01", "2025-12-31", True),    # Ball Corp, ativo
    ("CDAY", "2021-07-01", "2025-12-31", True),    # Ceridian, ativo
    ("GPS",  "2015-07-01", "2025-12-31", True),    # Gap, ativo
    ("MMC",  "2015-07-01", "2025-12-31", True),    # Marsh McLennan, ativo
    ("RE",   "2015-07-01", "2025-12-31", True),    # Everest Re, ativo
    # --- Grupo 3: renomeados (mesmo ticker até a data de saída) ---
    ("ANTM", "2015-07-01", "2022-06-28", False),   # → ELV jun/2022
    ("FBHS", "2015-07-01", "2022-11-08", False),   # → FBIN nov/2022
    ("FRC",  "2018-07-01", "2023-05-01", False),   # falência mai/2023
    ("PEAK", "2019-07-01", "2024-02-01", False),   # → DOC fev/2024
    ("PKI",  "2015-07-01", "2023-05-04", False),   # → RVTY mar/2023
    ("TMK",  "2015-07-01", "2019-08-08", False),   # → GL ago/2019
    ("WRK",  "2015-07-01", "2024-07-05", False),   # → SW jul/2024
    # --- Parcial: falta 2016-2017 ---
    ("IR",   "2015-07-01", "2017-05-11", True),    # old Ingersoll-Rand (ainda existe como IR)
]

print(f"Targets: {len(targets_b)} tickers")

Targets: 17 tickers


In [11]:
def fetch_yahoo_b(ticker: str, start: str, end: str) -> pd.Series:
    """
    Usa Ticker.history() em vez de download() — evita o bug 'no timezone found'
    de certas versões do yfinance. auto_adjust=False → Close split-only (sem dividendos).
    """
    try:
        t = yf.Ticker(ticker)
        raw = t.history(start=start, end=end, auto_adjust=False)
        if raw.empty or "Close" not in raw.columns:
            return pd.Series(dtype=float, name=ticker)

        close = raw["Close"].copy()
        close.index = pd.to_datetime(close.index).tz_localize(None)
        return close.dropna().rename(ticker)
    except Exception as e:
        print(f"    yahoo error: {e}")
        return pd.Series(dtype=float, name=ticker)


def fetch_tiingo_b(ticker: str, start: str, end: str) -> pd.Series:
    """Mesmo padrão do pipeline_base Parte 2: campo close (sem dividendos)."""
    try:
        r = requests.get(
            f"https://api.tiingo.com/tiingo/daily/{ticker}/prices",
            params={"startDate": start, "endDate": end,
                    "token": TIINGO_TOKEN, "resampleFreq": "daily"},
            timeout=15
        )
        if r.status_code != 200:
            print(f"    tiingo HTTP {r.status_code}")
            return pd.Series(dtype=float, name=ticker)

        data = r.json()
        if not data:
            return pd.Series(dtype=float, name=ticker)

        df = pd.DataFrame(data)
        df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
        return df.set_index("date")["close"].dropna().rename(ticker)
    except Exception as e:
        print(f"    tiingo error: {e}")
        return pd.Series(dtype=float, name=ticker)


def save_recovery(ticker: str, s: pd.Series):
    out = s.reset_index()
    out.columns = ["Date", ticker]
    out.to_csv(RECOVERY_DIR / f"{ticker}.csv", index=False)


print("Funções carregadas.")

Funções carregadas.


In [12]:
results_b = {}

for ticker, t_start, t_end, yahoo_ativo in targets_b:
    print(f"\n[{ticker}]", end="  ")
    s = pd.Series(dtype=float)

    # Passo 1: Yahoo (só para tickers ainda ativos)
    if yahoo_ativo:
        s = fetch_yahoo_b(ticker, t_start, t_end)
        if len(s) > 10:
            save_recovery(ticker, s)
            print(f"✅ Yahoo  {len(s)} linhas  ({s.index[0].date()} → {s.index[-1].date()})")
            results_b[ticker] = "yahoo"
            time.sleep(0.3)
            continue
        else:
            print(f"Yahoo ❌ ({len(s)} linhas) →", end="  ")

    # Passo 2: Tiingo com datas exatas de membership
    s = fetch_tiingo_b(ticker, t_start, t_end)
    if len(s) > 10:
        save_recovery(ticker, s)
        print(f"✅ Tiingo  {len(s)} linhas  ({s.index[0].date()} → {s.index[-1].date()})")
        results_b[ticker] = "tiingo"
    else:
        print(f"Tiingo ❌ ({len(s)} linhas)")
        results_b[ticker] = "fail"

    time.sleep(0.5)

# Resumo
print(f"\n{'='*50}")
ok   = [t for t, v in results_b.items() if v != "fail"]
fail = [t for t, v in results_b.items() if v == "fail"]
print(f"✅ Recuperados ({len(ok)}): {ok}")
print(f"❌ Falharam   ({len(fail)}): {fail}")


[CBS]      tiingo HTTP 404
Tiingo ❌ (0 linhas)

[ENDP]      tiingo HTTP 404
Tiingo ❌ (0 linhas)

[MNK]      tiingo HTTP 404
Tiingo ❌ (0 linhas)

[APC]  Tiingo ❌ (0 linhas)

[BLL]  

$BLL: possibly delisted; no timezone found


Yahoo ❌ (0 linhas) →  Tiingo ❌ (0 linhas)


$CDAY: possibly delisted; no timezone found



[CDAY]  Yahoo ❌ (0 linhas) →      tiingo HTTP 404
Tiingo ❌ (0 linhas)


...


[MMC]  Yahoo ❌ (0 linhas) →  Tiingo ❌ (0 linhas)


$RE: possibly delisted; no timezone found



[RE]  Yahoo ❌ (0 linhas) →  Tiingo ❌ (0 linhas)

[ANTM]  Tiingo ❌ (0 linhas)

[FBHS]  Tiingo ❌ (0 linhas)

[FRC]      tiingo HTTP 404
Tiingo ❌ (0 linhas)

[PEAK]  Tiingo ❌ (0 linhas)

[PKI]  Tiingo ❌ (0 linhas)

[TMK]  Tiingo ❌ (0 linhas)

[WRK]  Tiingo ❌ (1 linhas)


$IR: possibly delisted; no price data found  (1d 2015-07-01 -> 2017-05-11) (Yahoo error = "Data doesn't exist for startDate = 1435723200, endDate = 1494475200")



[IR]  Yahoo ❌ (0 linhas) →  Tiingo ❌ (0 linhas)

✅ Recuperados (0): []
❌ Falharam   (17): ['CBS', 'ENDP', 'MNK', 'APC', 'BLL', 'CDAY', 'GPS', 'MMC', 'RE', 'ANTM', 'FBHS', 'FRC', 'PEAK', 'PKI', 'TMK', 'WRK', 'IR']


---
## Parte C — Financial Data API + Financial Modeling Prep

Dois novos provedores com API key própria.

In [16]:
import requests, time
import pandas as pd
from pathlib import Path

FINANCIALDATA_KEY = "ff6ca5dffb9951ff93400949958695de"
RECOVERY_DIR      = Path("../data_bases/external/macrotrends_recovery")
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

# Todos os targets não-reciclados (mesmas datas de membership da Parte B)
targets_fd = [
    ("CBS",  "2015-07-01", "2019-12-05"),
    ("ENDP", "2015-07-01", "2022-08-16"),
    ("MNK",  "2015-07-01", "2020-10-12"),
    ("APC",  "2015-07-01", "2019-08-08"),
    ("BLL",  "2015-07-01", "2025-12-31"),
    ("CDAY", "2021-07-01", "2025-12-31"),
    ("GPS",  "2015-07-01", "2025-12-31"),
    ("MMC",  "2015-07-01", "2025-12-31"),
    ("RE",   "2015-07-01", "2025-12-31"),
    ("ANTM", "2015-07-01", "2022-06-28"),
    ("FBHS", "2015-07-01", "2022-11-08"),
    ("FRC",  "2018-07-01", "2023-05-01"),
    ("PEAK", "2019-07-01", "2024-02-01"),
    ("PKI",  "2015-07-01", "2023-05-04"),
    ("TMK",  "2015-07-01", "2019-08-08"),
    ("WRK",  "2015-07-01", "2024-07-05"),
    ("IR",   "2015-07-01", "2017-05-11"),
]


def fetch_financialdata(ticker: str, start: str, end: str) -> pd.Series:
    """
    Busca todos os registros paginando com offset até cobrir o período.
    A API retorna 300 registros por chamada, do mais recente para o mais antigo.
    """
    records = []
    offset  = 0
    start_dt = pd.to_datetime(start)

    while True:
        try:
            r = requests.get(
                "https://financialdata.net/api/v1/stock-prices",
                params={"identifier": ticker, "key": FINANCIALDATA_KEY,
                        "format": "json", "offset": offset},
                timeout=15)
        except Exception as e:
            print(f"    connection error: {e}")
            break

        if not r.ok:
            print(f"    HTTP {r.status_code}")
            break

        page = r.json()
        if not isinstance(page, list) or len(page) == 0:
            break  # sem mais dados

        records.extend(page)

        # Data mais antiga desta página
        oldest = pd.to_datetime(page[-1]["date"])
        if oldest <= start_dt:
            break  # chegamos ao período que precisamos

        offset += 300
        time.sleep(0.4)

    if not records:
        return pd.Series(dtype=float, name=ticker)

    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"])
    df = df[(df["date"] >= start) & (df["date"] <= end)]
    df = df.sort_values("date").drop_duplicates("date")

    return df.set_index("date")["close"].rename(ticker)


# ── Coleta ──
results_fd = {}

for ticker, t_start, t_end in targets_fd:
    print(f"[{ticker}]  ", end="", flush=True)
    s = fetch_financialdata(ticker, t_start, t_end)

    if len(s) > 10:
        out = s.reset_index()
        out.columns = ["Date", ticker]
        out.to_csv(RECOVERY_DIR / f"{ticker}.csv", index=False)
        print(f"✅  {len(s)} linhas  ({s.index[0].date()} → {s.index[-1].date()})")
        results_fd[ticker] = "ok"
    else:
        print(f"❌  {len(s)} linhas — sem dados")
        results_fd[ticker] = "fail"

    time.sleep(0.5)

print(f"\n{'='*50}")
ok   = [t for t, v in results_fd.items() if v == "ok"]
fail = [t for t, v in results_fd.items() if v == "fail"]
print(f"✅ Recuperados ({len(ok)}): {ok}")
print(f"❌ Falharam   ({len(fail)}): {fail}")

[CBS]  ❌  0 linhas — sem dados
[ENDP]  ❌  0 linhas — sem dados
[MNK]  ❌  0 linhas — sem dados
[APC]  ❌  0 linhas — sem dados
[BLL]  ❌  0 linhas — sem dados
[CDAY]  ❌  0 linhas — sem dados
[GPS]  ❌  0 linhas — sem dados
[MMC]  ✅  2596 linhas  (2015-09-04 → 2025-12-31)
[RE]  ❌  0 linhas — sem dados
[ANTM]  ❌  0 linhas — sem dados
[FBHS]  ❌  0 linhas — sem dados
[FRC]  ❌  0 linhas — sem dados
[PEAK]  ❌  0 linhas — sem dados
[PKI]  ❌  0 linhas — sem dados
[TMK]  ❌  0 linhas — sem dados
[WRK]  ❌  0 linhas — sem dados
[IR]  ❌  0 linhas — sem dados

✅ Recuperados (1): ['MMC']
❌ Falharam   (16): ['CBS', 'ENDP', 'MNK', 'APC', 'BLL', 'CDAY', 'GPS', 'RE', 'ANTM', 'FBHS', 'FRC', 'PEAK', 'PKI', 'TMK', 'WRK', 'IR']


In [17]:
# ── Validação: Financial Data `close` vs Yahoo Close existente ──
# Busca AAPL nas duas fontes e calcula ratio (deve ser ≈ 1.0 se o padrão for igual)

import pandas as pd, requests, time
from pathlib import Path

FINANCIALDATA_KEY = "ff6ca5dffb9951ff93400949958695de"
PRICES_DIR = Path("../data_bases/prices")

# 1) AAPL do Financial Data API — pegar duas páginas para ter 2023-2024
records_aapl = []
for offset in [300, 600]:   # offset 300 ≈ 2024-2025, offset 600 ≈ 2023-2024
    r = requests.get("https://financialdata.net/api/v1/stock-prices",
                     params={"identifier": "AAPL", "key": FINANCIALDATA_KEY,
                             "format": "json", "offset": offset}, timeout=15)
    records_aapl.extend(r.json())
    time.sleep(0.4)

fd_aapl = pd.DataFrame(records_aapl)
fd_aapl["date"] = pd.to_datetime(fd_aapl["date"])
fd_aapl = fd_aapl.set_index("date")["close"].sort_index()
print(f"Financial Data AAPL: {len(fd_aapl)} registros | {fd_aapl.index[0].date()} → {fd_aapl.index[-1].date()}")
print(f"  Exemplos: {fd_aapl.iloc[:3].to_dict()}")

# 2) AAPL do arquivo existente (Yahoo)
yahoo_aapl = pd.read_csv(PRICES_DIR / "AAPL.csv", index_col=0, parse_dates=True)
yahoo_aapl.index = pd.to_datetime(yahoo_aapl.index).tz_localize(None)
yahoo_aapl = yahoo_aapl.iloc[:, 0].sort_index()
print(f"\nYahoo AAPL (existente): {len(yahoo_aapl)} registros | {yahoo_aapl.index[0].date()} → {yahoo_aapl.index[-1].date()}")

# 3) Comparar datas em comum
common = fd_aapl.index.intersection(yahoo_aapl.index)
cmp = pd.DataFrame({"fd": fd_aapl[common], "yahoo": yahoo_aapl[common]}).dropna()
cmp["ratio"] = cmp["fd"] / cmp["yahoo"]

print(f"\n── Comparação em {len(cmp)} datas em comum ──")
print(cmp.head(5).to_string())
print(f"\nRatio: mean={cmp['ratio'].mean():.6f}  std={cmp['ratio'].std():.6f}  "
      f"min={cmp['ratio'].min():.4f}  max={cmp['ratio'].max():.4f}")
if cmp["ratio"].std() < 0.001:
    print("✅ Financial Data close = Yahoo Close (mesmo padrão — split-adjusted, sem dividendos)")
else:
    print("⚠️  Valores divergem — verificar se é adjClose ou close diferente")

Financial Data AAPL: 600 registros | 2022-09-26 → 2025-02-14
  Exemplos: {Timestamp('2022-09-26 00:00:00'): 150.77, Timestamp('2022-09-27 00:00:00'): 151.76, Timestamp('2022-09-28 00:00:00'): 149.84}

Yahoo AAPL (existente): 2641 registros | 2015-07-01 → 2025-12-30

── Comparação em 600 datas em comum ──
                fd       yahoo  ratio
2022-09-26  150.77  150.770004    1.0
2022-09-27  151.76  151.759995    1.0
2022-09-28  149.84  149.839996    1.0
2022-09-29  142.48  142.479996    1.0
2022-09-30  138.20  138.199997    1.0

Ratio: mean=1.000000  std=0.000002  min=1.0000  max=1.0000
✅ Financial Data close = Yahoo Close (mesmo padrão — split-adjusted, sem dividendos)


In [22]:
import shutil, pandas as pd
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")

def incorporate_recovered(ticker: str, dry_run: bool = True):
    rec_file   = RECOVERY_DIR / f"{ticker}.csv"
    price_file = PRICES_DIR   / f"{ticker}.csv"

    if not rec_file.exists():
        print(f"[{ticker}] Sem arquivo em recovery/")
        return

    rec_df = pd.read_csv(rec_file, parse_dates=["Date"])

    if price_file.exists():
        existing = pd.read_csv(price_file, parse_dates=["Date"])
        merged = (pd.concat([rec_df, existing])
                  .drop_duplicates("Date")
                  .sort_values("Date")
                  .reset_index(drop=True))
        action = f"merge: {len(rec_df)} novas + {len(existing)} existentes = {len(merged)} total"
        if not dry_run:
            merged.to_csv(price_file, index=False)
    else:
        merged = rec_df
        action = f"cópia nova: {len(merged)} linhas"
        if not dry_run:
            shutil.copy(rec_file, price_file)

    status = "[DRY RUN]" if dry_run else "[GRAVADO]"
    print(f"{status} [{ticker}] {action}")
    if not dry_run:
        print(f"  Verificação: {len(pd.read_csv(price_file))} linhas em prices/{ticker}.csv")

# Preview primeiro
incorporate_recovered("MMC", dry_run=False)

# Descomentar para gravar:
# incorporate_recovered("MMC", dry_run=False)

[GRAVADO] [MMC] cópia nova: 2596 linhas
  Verificação: 2596 linhas em prices/MMC.csv


---
## Parte D — Twelve Data (`include_delisted=true`)

In [23]:
import requests, json, time
import pandas as pd

TWELVE_KEY = "57426acae82248c2911f7064b998e3d3"
BASE = "https://api.twelvedata.com/time_series"

def probe_twelve(ticker, start="2018-01-01", end="2019-01-10"):
    """Teste rápido: retorna quantas linhas e os campos disponíveis."""
    r = requests.get(BASE, params={
        "symbol": ticker,
        "interval": "1day",
        "start_date": start,
        "end_date": end,
        "include_delisted": "true",
        "apikey": TWELVE_KEY,
    }, timeout=15)
    if not r.ok:
        print(f"  [{ticker}] HTTP {r.status_code}")
        return
    d = r.json()
    status = d.get("status", "?")
    if status != "ok":
        print(f"  [{ticker}] status={status} | {d.get('message', d.get('code', ''))}")
        return
    values = d.get("values", [])
    meta   = d.get("meta", {})
    print(f"  [{ticker}] ✅ {len(values)} registros | exchange={meta.get('exchange')} | "
          f"type={meta.get('type')} | {values[-1]['datetime'] if values else '?'} → {values[0]['datetime'] if values else '?'}")
    if values:
        print(f"    campos: {list(values[0].keys())}")
        print(f"    ex: {values[0]}")

# Diagnóstico com 4 casos representativos
print("=== TWELVE DATA — diagnóstico ===\n")
# Ativo (deveria ser trivial)
probe_twelve("BLL",  "2020-01-01", "2020-01-15")
time.sleep(1)
# Delistado por fusão
probe_twelve("CBS",  "2018-01-01", "2019-01-10")
time.sleep(1)
# Renomeado (mesmo CUSIP, ticker diferente)
probe_twelve("ANTM", "2016-01-01", "2016-01-15")
time.sleep(1)
# Adquirida
probe_twelve("APC",  "2016-01-01", "2019-08-08")
time.sleep(1)
# Ticker com sufixo de exchange (fallback se o simples falhar)
probe_twelve("CBS:NYSE", "2018-01-01", "2019-01-10")

=== TWELVE DATA — diagnóstico ===

  [BLL] status=error | This symbol is available starting with the Pro or Venture plan. Consider upgrading now at https://twelvedata.com/pricing
  [CBS] status=error | **symbol** or **figi** parameter is missing or invalid. Please provide a valid symbol according to API documentation: https://twelvedata.com/docs#reference-data
  [ANTM] status=error | This symbol is available starting with the Grow or Venture plan. Consider upgrading now at https://twelvedata.com/pricing
  [APC] ✅ 906 registros | exchange=NYSE | type=Common Stock | 2016-01-03 → 2019-08-07
    campos: ['datetime', 'open', 'high', 'low', 'close', 'volume']
    ex: {'datetime': '2019-08-07', 'open': '72.40000', 'high': '72.95000', 'low': '72.12000', 'close': '72.77000', 'volume': '44791048'}
  [CBS:NYSE] status=error | **symbol** or **figi** parameter is missing or invalid. Please provide a valid symbol according to API documentation: https://twelvedata.com/docs#reference-data


In [24]:
import requests, time, shutil
import pandas as pd
from pathlib import Path

TWELVE_KEY   = "57426acae82248c2911f7064b998e3d3"
BASE         = "https://api.twelvedata.com/time_series"
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")
RECOVERY_DIR.mkdir(parents=True, exist_ok=True)

targets_twelve = [
    ("CBS",  "2015-07-01", "2019-12-05"),
    ("ENDP", "2015-07-01", "2022-08-16"),
    ("MNK",  "2015-07-01", "2020-10-12"),
    ("APC",  "2015-07-01", "2019-08-08"),
    ("BLL",  "2015-07-01", "2025-12-31"),
    ("CDAY", "2021-07-01", "2025-12-31"),
    ("GPS",  "2015-07-01", "2025-12-31"),
    ("RE",   "2015-07-01", "2025-12-31"),
    ("ANTM", "2015-07-01", "2022-06-28"),
    ("FBHS", "2015-07-01", "2022-11-08"),
    ("FRC",  "2018-07-01", "2023-05-01"),
    ("PEAK", "2019-07-01", "2024-02-01"),
    ("PKI",  "2015-07-01", "2023-05-04"),
    ("TMK",  "2015-07-01", "2019-08-08"),
    ("WRK",  "2015-07-01", "2024-07-05"),
    ("IR",   "2015-07-01", "2017-05-11"),
]


def fetch_twelve(ticker: str, start: str, end: str) -> pd.Series:
    """
    Twelve Data retorna no máximo 5000 registros por chamada.
    Para períodos mais longos, pagina via `end_date` decrescente.
    """
    all_records = []
    current_end = end

    while True:
        r = requests.get(BASE, params={
            "symbol": ticker,
            "interval": "1day",
            "start_date": start,
            "end_date": current_end,
            "outputsize": 5000,
            "include_delisted": "true",
            "apikey": TWELVE_KEY,
        }, timeout=15)

        if not r.ok:
            break
        d = r.json()
        if d.get("status") != "ok":
            # Guardar mensagem de erro para exibir
            fetch_twelve._last_err = d.get("message", d.get("code", "erro desconhecido"))
            break

        values = d.get("values", [])
        if not values:
            break

        all_records.extend(values)

        # Verificar se já cobrimos o período todo
        oldest = pd.to_datetime(values[-1]["datetime"])
        if oldest <= pd.to_datetime(start):
            break

        # Próxima página: end_date = dia anterior ao mais antigo desta página
        current_end = (oldest - pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        time.sleep(0.5)

    if not all_records:
        return pd.Series(dtype=float, name=ticker)

    df = pd.DataFrame(all_records)
    df["date"] = pd.to_datetime(df["datetime"])
    df["close"] = pd.to_numeric(df["close"], errors="coerce")
    df = df[(df["date"] >= start) & (df["date"] <= end)]
    df = df.sort_values("date").drop_duplicates("date")
    return df.set_index("date")["close"].rename(ticker)


fetch_twelve._last_err = ""

# ── Coleta ──
results_twelve = {}

for ticker, t_start, t_end in targets_twelve:
    fetch_twelve._last_err = ""
    print(f"[{ticker}]  ", end="", flush=True)
    s = fetch_twelve(ticker, t_start, t_end)

    if len(s) > 10:
        out = s.reset_index()
        out.columns = ["Date", ticker]
        out.to_csv(RECOVERY_DIR / f"{ticker}.csv", index=False)
        print(f"✅  {len(s)} linhas  ({s.index[0].date()} → {s.index[-1].date()})")
        results_twelve[ticker] = "ok"
    else:
        err = fetch_twelve._last_err[:80] if fetch_twelve._last_err else "sem dados"
        print(f"❌  {err}")
        results_twelve[ticker] = "fail"

    time.sleep(1)

print(f"\n{'='*55}")
ok   = [t for t, v in results_twelve.items() if v == "ok"]
fail = [t for t, v in results_twelve.items() if v == "fail"]
print(f"✅ Recuperados ({len(ok)}): {ok}")
print(f"❌ Falharam   ({len(fail)}): {fail}")

[CBS]  ❌  **symbol** or **figi** parameter is missing or invalid. Please provide a valid s
[ENDP]  ❌  This symbol is available starting with the Ultra or Enterprise plan. Consider up
[MNK]  ❌  This symbol is available starting with the Grow or Venture plan. Consider upgrad
[APC]  ✅  1033 linhas  (2015-07-01 → 2019-08-07)
[BLL]  ❌  This symbol is available starting with the Pro or Venture plan. Consider upgradi
[CDAY]  ❌  This symbol is available starting with the Grow or Venture plan. Consider upgrad
[GPS]  ✅  1311 linhas  (2015-07-01 → 2025-12-30)
[RE]  ❌  This symbol is available starting with the Grow or Venture plan. Consider upgrad
[ANTM]  ❌  You have run out of API credits for the current minute. 9 API credits were used,
[FBHS]  ❌  You have run out of API credits for the current minute. 10 API credits were used
[FRC]  ❌  You have run out of API credits for the current minute. 11 API credits were used
[PEAK]  ❌  You have run out of API credits for the current minute. 12 API credit

In [25]:
# ── Retry dos que bateram no rate limit + validação de GPS ──
# Free tier Twelve Data: 8 créditos/minuto → sleep de 10s entre chamadas

retry_targets = [
    ("ANTM", "2015-07-01", "2022-06-28"),
    ("FBHS", "2015-07-01", "2022-11-08"),
    ("FRC",  "2018-07-01", "2023-05-01"),
    ("PEAK", "2019-07-01", "2024-02-01"),
]

print("=== RETRY (10s entre chamadas) ===\n")
for ticker, t_start, t_end in retry_targets:
    fetch_twelve._last_err = ""
    print(f"[{ticker}]  ", end="", flush=True)
    s = fetch_twelve(ticker, t_start, t_end)
    if len(s) > 10:
        out = s.reset_index()
        out.columns = ["Date", ticker]
        out.to_csv(RECOVERY_DIR / f"{ticker}.csv", index=False)
        print(f"✅  {len(s)} linhas  ({s.index[0].date()} → {s.index[-1].date()})")
        results_twelve[ticker] = "ok"
    else:
        err = fetch_twelve._last_err[:80] if fetch_twelve._last_err else "sem dados"
        print(f"❌  {err}")
        results_twelve[ticker] = "fail"
    print(f"  aguardando 10s...", flush=True)
    time.sleep(10)

# ── Validação rápida: GPS e APC ──
print("\n=== VALIDAÇÃO DE PREÇOS ===\n")

gps = pd.read_csv(RECOVERY_DIR / "GPS.csv", parse_dates=["Date"])
gps = gps.set_index("Date")["GPS"]

apc = pd.read_csv(RECOVERY_DIR / "APC.csv", parse_dates=["Date"])
apc = apc.set_index("Date")["APC"]

print("GPS (Gap Inc.) — sanity check:")
for date_str in ["2016-01-04", "2020-03-31", "2022-01-03", "2024-01-02"]:
    dt = pd.to_datetime(date_str)
    if dt in gps.index:
        print(f"  {date_str}: ${gps[dt]:.2f}")
    else:
        nearest = gps.index[gps.index.get_indexer([dt], method='nearest')[0]]
        print(f"  {date_str} (nearest {nearest.date()}): ${gps[nearest]:.2f}")

print("\nAPC (Anadarko Petroleum) — sanity check:")
for date_str in ["2016-01-04", "2018-01-02", "2019-08-07"]:
    dt = pd.to_datetime(date_str)
    if dt in apc.index:
        print(f"  {date_str}: ${apc[dt]:.2f}")
    else:
        nearest = apc.index[apc.index.get_indexer([dt], method='nearest')[0]]
        print(f"  {date_str} (nearest {nearest.date()}): ${apc[nearest]:.2f}")

=== RETRY (10s entre chamadas) ===

[ANTM]  ❌  This symbol is available starting with the Grow or Venture plan. Consider upgrad
  aguardando 10s...
[FBHS]  ✅  1853 linhas  (2015-07-01 → 2022-11-07)
  aguardando 10s...
[FRC]  ❌  This symbol is available starting with the Pro or Venture plan. Consider upgradi
  aguardando 10s...
[PEAK]  ❌  This symbol is available starting with the Grow or Venture plan. Consider upgrad
  aguardando 10s...

=== VALIDAÇÃO DE PREÇOS ===

GPS (Gap Inc.) — sanity check:
  2016-01-04: $25.51
  2020-03-31 (nearest 2019-11-19): $16.78
  2022-01-03 (nearest 2019-11-19): $16.78
  2024-01-02 (nearest 2025-03-10): $22.36

APC (Anadarko Petroleum) — sanity check:
  2016-01-04: $48.52
  2018-01-02: $54.77
  2019-08-07: $72.77


In [27]:
import pandas as pd, requests, time
from pathlib import Path

TWELVE_KEY   = "57426acae82248c2911f7064b998e3d3"
BASE         = "https://api.twelvedata.com/time_series"
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")
PRICES_DIR   = Path("../data_bases/prices")


def check_coverage(ticker, membership_end):
    """Mostra registros por ano e se o membership está coberto."""
    path = RECOVERY_DIR / f"{ticker}.csv"
    if not path.exists():
        print(f"[{ticker}] arquivo não encontrado")
        return
    s = pd.read_csv(path, parse_dates=["Date"]).set_index("Date").iloc[:, 0]
    by_year = s.groupby(s.index.year).count()
    end_dt  = pd.to_datetime(membership_end)
    covered = s.index.max() >= end_dt
    print(f"\n[{ticker}] {len(s)} linhas | {s.index[0].date()} → {s.index[-1].date()}")
    print(f"  Por ano: {by_year.to_dict()}")
    print(f"  Cobre membership até {membership_end}? {'✅ SIM' if covered else '❌ NÃO (falta dados após ' + str(s.index.max().date()) + ')'}")

check_coverage("GPS",  "2022-01-10")   # membership termina jan/2022
check_coverage("FBHS", "2022-11-08")   # membership termina nov/2022
check_coverage("APC",  "2019-08-08")   # membership termina ago/2019

# ── Ratio test: AAPL Twelve Data vs Yahoo ──
print("\n── Ratio test AAPL (Twelve Data vs Yahoo) ──")
print("aguardando 10s (rate limit)...", flush=True)
time.sleep(10)

r = requests.get(BASE, params={
    "symbol": "AAPL", "interval": "1day",
    "start_date": "2023-01-01", "end_date": "2024-06-30",
    "outputsize": 400, "apikey": TWELVE_KEY,
}, timeout=15)
d = r.json()
if d.get("status") == "ok":
    td_aapl = (pd.DataFrame(d["values"])
               .assign(Date=lambda x: pd.to_datetime(x["datetime"]))
               .set_index("Date")["close"].astype(float).sort_index())

    yh_aapl = (pd.read_csv(PRICES_DIR / "AAPL.csv", index_col=0, parse_dates=True)
               .pipe(lambda df: df.set_index(pd.to_datetime(df.index).tz_localize(None)))
               .iloc[:, 0].sort_index())

    common = td_aapl.index.intersection(yh_aapl.index)
    cmp = pd.DataFrame({"twelve": td_aapl[common], "yahoo": yh_aapl[common]}).dropna()
    cmp["ratio"] = cmp["twelve"] / cmp["yahoo"]
    print(cmp.head(4).to_string())
    print(f"\nRatio: mean={cmp['ratio'].mean():.6f}  std={cmp['ratio'].std():.6f}")
    if cmp["ratio"].std() < 0.001:
        print("✅ Twelve Data close = Yahoo Close (mesmo padrão)")
    else:
        print("⚠️  Valores divergem — verificar ajuste")
else:
    print("Erro:", d.get("message", "?"))


[GPS] 1311 linhas | 2015-07-01 → 2025-12-30
  Por ano: {2015: 128, 2016: 252, 2017: 251, 2018: 251, 2019: 224, 2025: 205}
  Cobre membership até 2022-01-10? ✅ SIM

[FBHS] 1853 linhas | 2015-07-01 → 2022-11-07
  Por ano: {2015: 128, 2016: 252, 2017: 251, 2018: 251, 2019: 252, 2020: 253, 2021: 252, 2022: 214}
  Cobre membership até 2022-11-08? ❌ NÃO (falta dados após 2022-11-07)

[APC] 1033 linhas | 2015-07-01 → 2019-08-07
  Por ano: {2015: 127, 2016: 252, 2017: 251, 2018: 251, 2019: 152}
  Cobre membership até 2019-08-08? ❌ NÃO (falta dados após 2019-08-07)

── Ratio test AAPL (Twelve Data vs Yahoo) ──
aguardando 10s (rate limit)...
                twelve       yahoo  ratio
Date                                     
2023-01-03  125.070000  125.070000    1.0
2023-01-04  126.360000  126.360001    1.0
2023-01-05  125.019997  125.019997    1.0
2023-01-06  129.620000  129.619995    1.0

Ratio: mean=1.000000  std=0.000000
✅ Twelve Data close = Yahoo Close (mesmo padrão)


In [28]:
import shutil, pandas as pd
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")

def incorporate(ticker: str, dry_run: bool = False):
    rec   = RECOVERY_DIR / f"{ticker}.csv"
    dest  = PRICES_DIR   / f"{ticker}.csv"
    if not rec.exists():
        print(f"[{ticker}] sem arquivo em recovery/")
        return

    rec_df = pd.read_csv(rec, parse_dates=["Date"])

    if dest.exists():
        existing = pd.read_csv(dest, parse_dates=["Date"])
        merged = (pd.concat([rec_df, existing])
                  .drop_duplicates("Date").sort_values("Date").reset_index(drop=True))
        action = f"merge: {len(rec_df)} novas + {len(existing)} existentes = {len(merged)} total"
    else:
        merged = rec_df
        action = f"novo: {len(merged)} linhas"

    if not dry_run:
        merged.to_csv(dest, index=False)
        n = len(pd.read_csv(dest))
        print(f"✅ [{ticker}] {action} → prices/{ticker}.csv ({n} linhas gravadas)")
    else:
        print(f"[DRY RUN] [{ticker}] {action}")

# Incorporar os 3 recuperados
for t in ["FBHS", "APC", "GPS"]:
    incorporate(t)

✅ [FBHS] novo: 1853 linhas → prices/FBHS.csv (1853 linhas gravadas)
✅ [APC] novo: 1033 linhas → prices/APC.csv (1033 linhas gravadas)
✅ [GPS] novo: 1311 linhas → prices/GPS.csv (1311 linhas gravadas)


In [1]:
import yfinance as yf

# EG tem histórico completo (mesmo company, novo ticker)?
t = yf.Ticker("EG")
hist = t.history(start="2017-01-01", end="2023-06-30", auto_adjust=False)
print(f"EG: {len(hist)} linhas | {hist.index[0].date()} → {hist.index[-1].date()}")
print(hist["Close"].head(3))


EG: 1633 linhas | 2017-01-03 → 2023-06-29
Date
2017-01-03 00:00:00-05:00    216.039993
2017-01-04 00:00:00-05:00    217.800003
2017-01-05 00:00:00-05:00    217.779999
Name: Close, dtype: float64


---
## Parte E — Tickers renomeados via ticker atual no Yahoo

Insight de RE→EG: quando uma empresa muda de ticker, `yf.Ticker(novo_ticker).history()` retorna
o histórico completo desde antes da renomeação. Aplicando o mesmo padrão aos demais.

In [2]:
import yfinance as yf
import pandas as pd
import shutil
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")

# (orig_ticker, current_ticker, start, end)
# orig_ticker  = nome do arquivo CSV (como está no S&P 500 histórico)
# current_ticker = ticker que o Yahoo tem hoje (mesma empresa legal)
targets_e = [
    ("RE",   "EG",   "2017-01-01", "2023-06-30"),  # Everest Re → Everest Group
    ("ANTM", "ELV",  "2016-01-01", "2022-06-28"),  # Anthem → Elevance Health
    ("TMK",  "GL",   "2016-01-01", "2019-08-08"),  # Torchmark → Globe Life
    ("PKI",  "RVTY", "2016-01-01", "2023-05-04"),  # PerkinElmer → Revvity
    ("CDAY", "DAY",  "2021-07-01", "2024-02-01"),  # Ceridian → Dayforce
    ("PEAK", "DOC",  "2019-07-01", "2024-02-01"),  # Healthpeak → DOC (merged)
    ("WRK",  "SW",   "2016-01-01", "2024-07-05"),  # WestRock → Smurfit WestRock
    ("BLL",  "BLL",  "2016-01-01", "2022-04-11"),  # Ball Corp — sem rename, retry direto
]

results_e = {}

for orig, current, start, end in targets_e:
    try:
        t = yf.Ticker(current)
        raw = t.history(start=start, end=end, auto_adjust=False)
    except Exception as ex:
        print(f"[{orig}←{current}]  erro: {ex}")
        results_e[orig] = "fail"
        continue

    if raw.empty or "Close" not in raw.columns:
        print(f"[{orig}←{current}]  ❌ sem dados")
        results_e[orig] = "fail"
        continue

    close = raw["Close"].copy()
    close.index = pd.to_datetime(close.index).tz_localize(None)
    close = close.dropna()

    if len(close) > 10:
        out = close.rename(orig).reset_index()
        out.columns = ["Date", orig]
        out.to_csv(RECOVERY_DIR / f"{orig}.csv", index=False)
        print(f"[{orig}←{current}]  ✅  {len(close)} linhas  "
              f"({close.index[0].date()} → {close.index[-1].date()})")
        results_e[orig] = "ok"
    else:
        print(f"[{orig}←{current}]  ❌  {len(close)} linhas")
        results_e[orig] = "fail"

print(f"\n{'='*50}")
ok   = [t for t, v in results_e.items() if v == "ok"]
fail = [t for t, v in results_e.items() if v == "fail"]
print(f"✅ Recuperados ({len(ok)}): {ok}")
print(f"❌ Falharam   ({len(fail)}): {fail}")

[RE←EG]  ✅  1633 linhas  (2017-01-03 → 2023-06-29)
[ANTM←ELV]  ✅  1632 linhas  (2016-01-04 → 2022-06-27)
[TMK←GL]  ✅  905 linhas  (2016-01-04 → 2019-08-07)
[PKI←RVTY]  ✅  1846 linhas  (2016-01-04 → 2023-05-03)


$DAY: possibly delisted; no price data found  (1d 2021-07-01 -> 2024-02-01) (Yahoo error = "No data found, symbol may be delisted")


[CDAY←DAY]  ❌ sem dados
[PEAK←DOC]  ✅  1155 linhas  (2019-07-01 → 2024-01-31)
[WRK←SW]  ✅  2139 linhas  (2016-01-04 → 2024-07-03)


$BLL: possibly delisted; no timezone found


[BLL←BLL]  ❌ sem dados

✅ Recuperados (6): ['RE', 'ANTM', 'TMK', 'PKI', 'PEAK', 'WRK']
❌ Falharam   (2): ['CDAY', 'BLL']


In [3]:
import pandas as pd, shutil
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")

def incorporate(ticker: str):
    src  = RECOVERY_DIR / f"{ticker}.csv"
    dest = PRICES_DIR   / f"{ticker}.csv"
    if not src.exists():
        print(f"[{ticker}] sem arquivo em recovery/")
        return
    rec = pd.read_csv(src, parse_dates=["Date"])
    if dest.exists():
        existing = pd.read_csv(dest, parse_dates=["Date"])
        merged = (pd.concat([rec, existing])
                  .drop_duplicates("Date").sort_values("Date").reset_index(drop=True))
        merged.to_csv(dest, index=False)
        print(f"✅ [{ticker}] merge → {len(merged)} linhas  "
              f"({merged['Date'].min().date()} → {merged['Date'].max().date()})")
    else:
        shutil.copy(src, dest)
        print(f"✅ [{ticker}] novo  → {len(rec)} linhas  "
              f"({rec['Date'].min().date()} → {rec['Date'].max().date()})")

# Incorporar os 6 recuperados pela Parte E
for t in ["RE", "ANTM", "TMK", "PKI", "PEAK", "WRK"]:
    incorporate(t)

print("\nPróximo passo: re-rodar pipeline_base.ipynb a partir da Parte 4 para "
      "reconstruir extensao_2016_2025.csv e base_completa.csv com os novos dados.")

✅ [RE] novo  → 1633 linhas  (2017-01-03 → 2023-06-29)
✅ [ANTM] novo  → 1632 linhas  (2016-01-04 → 2022-06-27)
✅ [TMK] novo  → 905 linhas  (2016-01-04 → 2019-08-07)
✅ [PKI] novo  → 1846 linhas  (2016-01-04 → 2023-05-03)
✅ [PEAK] novo  → 1155 linhas  (2019-07-01 → 2024-01-31)
✅ [WRK] novo  → 2139 linhas  (2016-01-04 → 2024-07-03)

Próximo passo: re-rodar pipeline_base.ipynb a partir da Parte 4 para reconstruir extensao_2016_2025.csv e base_completa.csv com os novos dados.


---
## Parte F — Investing.com (download manual de CSV)

Formato: `Date,Price,Open,High,Low,Vol.,Change %` com datas MM/DD/YYYY e BOM UTF-8.

In [4]:
import pandas as pd
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")

# ── Parser para CSV do investing.com ──
def parse_investing(csv_path: str | Path, ticker: str) -> pd.Series:
    """
    Lê CSV do investing.com e retorna Series com index Date e nome=ticker.
    Lida com: BOM UTF-8, datas MM/DD/YYYY, vírgulas nos números ("1,234.56").
    """
    df = pd.read_csv(csv_path, encoding="utf-8-sig",
                     thousands=",", skipinitialspace=True)
    df.columns = df.columns.str.strip().str.strip('"')

    # Date está no formato MM/DD/YYYY
    df["Date"] = pd.to_datetime(df["Date"], format="%m/%d/%Y")

    # Coluna de preço de fechamento
    price_col = next(c for c in df.columns if c.strip() in ("Price", "price"))
    df[price_col] = pd.to_numeric(df[price_col].astype(str).str.replace(",", ""),
                                  errors="coerce")

    s = df.set_index("Date")[price_col].dropna().sort_index().rename(ticker)
    return s


# ── Validação: AAPL investing.com vs Financial Data API ──
# (nosso AAPL.csv vai até 2025-12-30; o CSV do Apple cobre abr-mai/2026)
# Usamos a comparação indireta: investing AAPL 04/28/2026 = 270.71
# Financial Data API AAPL 04/28/2026 = 270.71 (já validado ratio=1.0 com Yahoo)
# → confirma que investing.com "Price" = Yahoo Close (split-adjusted)

aapl_path = Path.home() / "Downloads" / "Apple Stock Price History.csv"
if aapl_path.exists():
    aapl_inv = parse_investing(aapl_path, "AAPL")
    print(f"AAPL investing.com: {len(aapl_inv)} linhas | "
          f"{aapl_inv.index[0].date()} → {aapl_inv.index[-1].date()}")
    print(aapl_inv.tail(3).to_string())
    # Verificar 04/28/2026 — deve ser 270.71 (mesmo que Financial Data API e Yahoo)
    d = pd.Timestamp("2026-04-28")
    if d in aapl_inv.index:
        val = aapl_inv[d]
        match = abs(val - 270.71) < 0.01
        print(f"\nAAPL 2026-04-28: {val:.2f} "
              f"{'✅ bate com Yahoo/Financial Data (270.71)' if match else '⚠️ diverge!'}")
else:
    print("Arquivo AAPL não encontrado em Downloads/ — ajuste o caminho abaixo.")
    print("Coloque o path correto em aapl_path e reexecute.")


# ── Parsear BLL ──
bll_path = Path.home() / "Downloads" / "Ball Stock Price History.csv"
if bll_path.exists():
    bll = parse_investing(bll_path, "BLL")
    print(f"\nBLL investing.com: {len(bll)} linhas | "
          f"{bll.index[0].date()} → {bll.index[-1].date()}")
    by_year = bll.groupby(bll.index.year).count()
    print(f"Por ano: {by_year.to_dict()}")
    print(f"\nNota: membership BLL = 2016/S1 → 2021/S2. "
          f"Faltam os anos: "
          f"{[y for y in range(2016, 2022) if y not in by_year.index]}")
else:
    print("\nArquivo BLL não encontrado em Downloads/ — ajuste o caminho.")
    print("Coloque o path correto em bll_path e reexecute.")

AAPL investing.com: 20 linhas | 2026-04-06 → 2026-05-01
Date
2026-04-29    270.17
2026-04-30    271.35
2026-05-01    280.14

AAPL 2026-04-28: 270.71 ✅ bate com Yahoo/Financial Data (270.71)

BLL investing.com: 251 linhas | 2021-01-05 → 2021-12-31
Por ano: {2021: 251}

Nota: membership BLL = 2016/S1 → 2021/S2. Faltam os anos: [2016, 2017, 2018, 2019, 2020]


In [9]:
import pandas as pd, shutil
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")
DOWNLOADS    = Path.home() / "Downloads"

# ── Liste aqui todos os CSVs do BLL baixados do investing.com ──
# Pode ser um arquivo único (período completo) ou vários (um por ano).
bll_files = sorted(DOWNLOADS.glob("Ball Stock Price History*.csv"))

if not bll_files:
    print("Nenhum arquivo BLL encontrado em Downloads/")
    print("Esperado: 'Ball Stock Price History.csv' ou 'Ball Stock Price History (1).csv' etc.")
else:
    print(f"Arquivos encontrados: {[f.name for f in bll_files]}")

    # Combinar todos os arquivos
    partes = [parse_investing(f, "BLL") for f in bll_files]
    bll_full = (pd.concat(partes)
                .sort_index()
                .drop_duplicates()
                [lambda s: (s.index >= "2016-01-01") & (s.index <= "2021-12-31")])

    by_year = bll_full.groupby(bll_full.index.year).count()
    print(f"\nBLL combinado: {len(bll_full)} linhas | "
          f"{bll_full.index[0].date()} → {bll_full.index[-1].date()}")
    print(f"Por ano: {by_year.to_dict()}")

    anos_faltando = [y for y in range(2016, 2022) if y not in by_year.index]
    if anos_faltando:
        print(f"\n⚠️  Ainda faltam: {anos_faltando} — baixe esses anos e reexecute.")
    else:
        print("\n✅ Cobertura completa 2016-2021!")

        # Salvar em recovery/ e incorporar
        out = bll_full.reset_index()
        out.columns = ["Date", "BLL"]
        out.to_csv(RECOVERY_DIR / "BLL.csv", index=False)
        print(f"Salvo em recovery/BLL.csv")

        # Incorporar em prices/
        dest = PRICES_DIR / "BLL.csv"
        if dest.exists():
            existing = pd.read_csv(dest, parse_dates=["Date"])
            merged = (pd.concat([out, existing])
                      .drop_duplicates("Date").sort_values("Date").reset_index(drop=True))
            merged.to_csv(dest, index=False)
            print(f"✅ BLL incorporado: merge → {len(merged)} linhas em prices/BLL.csv")
        else:
            shutil.copy(RECOVERY_DIR / "BLL.csv", dest)
            print(f"✅ BLL incorporado: {len(out)} linhas em prices/BLL.csv")

Arquivos encontrados: ['Ball Stock Price History (1).csv', 'Ball Stock Price History (2).csv', 'Ball Stock Price History (3).csv', 'Ball Stock Price History (4).csv', 'Ball Stock Price History (5).csv', 'Ball Stock Price History (6).csv', 'Ball Stock Price History.csv']

BLL combinado: 1225 linhas | 2016-01-04 → 2021-12-31
Por ano: {2016: 201, 2017: 171, 2018: 156, 2019: 242, 2020: 234, 2021: 221}

✅ Cobertura completa 2016-2021!
Salvo em recovery/BLL.csv
✅ BLL incorporado: merge → 1226 linhas em prices/BLL.csv


In [10]:
# ── Diagnóstico de gaps no BLL ──
import pandas as pd
from pathlib import Path

bll = pd.read_csv(Path("../data_bases/prices/BLL.csv"), parse_dates=["Date"])
bll = bll.set_index("Date")["BLL"].sort_index()

# Dias úteis NYSE esperados por mês (via calendário de negociações do MSFT)
msft = pd.read_csv(Path("../data_bases/prices/MSFT.csv"), parse_dates=["Date"])
msft = msft.set_index("Date")["MSFT"]
msft_mask = (msft.index >= "2016-01-01") & (msft.index <= "2021-12-31")
expected_by_month = msft[msft_mask].resample("MS").count().rename("esperado")

bll_by_month = bll.resample("MS").count().rename("bll")
cmp = pd.concat([expected_by_month, bll_by_month], axis=1).dropna()
cmp["falta"] = cmp["esperado"] - cmp["bll"]
gaps = cmp[cmp["falta"] > 3]  # meses com mais de 3 dias faltando

print(f"BLL: {len(bll)} linhas | MSFT (referência): {msft_mask.sum()} dias úteis")
print(f"Cobertura: {len(bll)/msft_mask.sum()*100:.1f}%")
print(f"\nMeses com gap > 3 dias ({len(gaps)} meses):")
if not gaps.empty:
    print(gaps.to_string())
else:
    print("Nenhum gap significativo ✅")

BLL: 1226 linhas | MSFT (referência): 1511 dias úteis
Cobertura: 81.1%

Meses com gap > 3 dias (32 meses):
            esperado  bll  falta
Date                            
2016-04-01        21   17      4
2016-05-01        21   15      6
2016-06-01        22   17      5
2016-07-01        20   14      6
2016-08-01        23   19      4
2016-10-01        21   14      7
2016-11-01        21   15      6
2016-12-01        21   14      7
2017-01-01        20   14      6
2017-02-01        19   15      4
2017-03-01        23   14      9
2017-04-01        19   10      9
2017-05-01        22   16      6
2017-08-01        23   15      8
2017-09-01        20   10     10
2017-10-01        22   16      6
2017-11-01        21   13      8
2017-12-01        20   10     10
2018-01-01        21   13      8
2018-02-01        19   11      8
2018-03-01        21   11     10
2018-04-01        21    9     12
2018-05-01        22   11     11
2018-06-01        21   12      9
2018-07-01        21    9     12
20

In [11]:
import pandas as pd, shutil
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")
DOWNLOADS    = Path.home() / "Downloads"

def parse_yahoo_web(csv_path, ticker: str) -> pd.Series:
    """CSV do site Yahoo Finance: Date,Open,High,Low,Close,Adj Close,Volume"""
    df = pd.read_csv(csv_path, parse_dates=["Date"])
    df = df[df["Close"] != "null"].copy()
    df["Close"] = pd.to_numeric(df["Close"], errors="coerce")
    return df.set_index("Date")["Close"].dropna().sort_index().rename(ticker)

def process_yahoo_web(ticker: str, start: str, end: str):
    path = DOWNLOADS / f"{ticker}.csv"
    if not path.exists():
        print(f"[{ticker}] nao encontrado em Downloads/")
        print(f"  -> https://finance.yahoo.com/quote/{ticker}/history/")
        print(f"     Time Period: {start} -> {end}  |  clique em Download")
        return
    s = parse_yahoo_web(path, ticker)
    s = s[(s.index >= start) & (s.index <= end)]
    msft = pd.read_csv(PRICES_DIR / "MSFT.csv", parse_dates=["Date"]).set_index("Date")["MSFT"]
    n_expected = ((msft.index >= start) & (msft.index <= end)).sum()
    cov = len(s) / n_expected * 100
    by_year = s.groupby(s.index.year).count()
    print(f"[{ticker}] {len(s)} linhas | {s.index[0].date()} -> {s.index[-1].date()}")
    print(f"  Por ano: {by_year.to_dict()}")
    print(f"  Cobertura: {cov:.1f}%")
    if cov > 95:
        out = s.reset_index(); out.columns = ["Date", ticker]
        out.to_csv(RECOVERY_DIR / f"{ticker}.csv", index=False)
        out.to_csv(PRICES_DIR   / f"{ticker}.csv", index=False)
        print(f"  OK Incorporado em prices/{ticker}.csv")
    else:
        print(f"  AVISO Cobertura {cov:.1f}% - ainda incompleto")

# BLL: membership 2016/S1 -> 2021/S2
process_yahoo_web("BLL",  "2016-01-01", "2021-12-31")
print()
# CDAY: membership 2021/S2 -> 2023/S2
process_yahoo_web("CDAY", "2021-09-01", "2023-12-31")


[BLL] nao encontrado em Downloads/
  -> https://finance.yahoo.com/quote/BLL/history/
     Time Period: 2016-01-01 -> 2021-12-31  |  clique em Download

[CDAY] nao encontrado em Downloads/
  -> https://finance.yahoo.com/quote/CDAY/history/
     Time Period: 2021-09-01 -> 2023-12-31  |  clique em Download


In [12]:
import yfinance as yf
import pandas as pd, shutil
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")

# BLL mudou para BALL em abr/2022 -- mesmo padrao RE->EG, ANTM->ELV, etc.
# yf.Ticker("BALL") tem historico completo incluindo periodo como BLL
raw = yf.Ticker("BALL").history(start="2016-01-01", end="2021-12-31", auto_adjust=False)
close = raw["Close"].copy()
close.index = pd.to_datetime(close.index).tz_localize(None)
close = close.dropna().sort_index()

msft = pd.read_csv(PRICES_DIR / "MSFT.csv", parse_dates=["Date"]).set_index("Date")["MSFT"]
n_expected = ((msft.index >= "2016-01-01") & (msft.index <= "2021-12-31")).sum()
cov = len(close) / n_expected * 100
by_year = close.groupby(close.index.year).count()

print(f"BALL->BLL: {len(close)} linhas | {close.index[0].date()} -> {close.index[-1].date()}")
print(f"Por ano: {by_year.to_dict()}")
print(f"Cobertura: {cov:.1f}%")

if cov > 95:
    out = close.rename("BLL").reset_index()
    out.columns = ["Date", "BLL"]
    out.to_csv(RECOVERY_DIR / "BLL.csv", index=False)
    out.to_csv(PRICES_DIR   / "BLL.csv", index=False)
    print("OK BLL incorporado de BALL com cobertura completa")

    # Verificar se BALL.csv ja existe (periodo pos-rename)
    ball_csv = PRICES_DIR / "BALL.csv"
    if ball_csv.exists():
        ball = pd.read_csv(ball_csv, parse_dates=["Date"])
        print(f"BALL.csv existente: {len(ball)} linhas | {ball["Date"].min().date()} -> {ball["Date"].max().date()}")
        print("  -> S&P500 provavelmente tem BALL como ticker ativo apos 2022")
else:
    print(f"AVISO Cobertura {cov:.1f}% - verificar")


BALL->BLL: 1510 linhas | 2016-01-04 -> 2021-12-30
Por ano: {2016: 252, 2017: 251, 2018: 251, 2019: 252, 2020: 253, 2021: 251}
Cobertura: 99.9%
OK BLL incorporado de BALL com cobertura completa
BALL.csv existente: 1002 linhas | 2022-01-03 -> 2025-12-30
  -> S&P500 provavelmente tem BALL como ticker ativo apos 2022


In [13]:
import yfinance as yf
import pandas as pd
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")

# CDAY (Ceridian HCM) renomeou para DAY (Dayforce) em fev/2024
# Membership CDAY: 2021/S2 -> 2023/S2 (todo periodo ainda sob ticker CDAY)
# yf.Ticker("DAY") deve ter historico desde 2021
raw = yf.Ticker("DAY").history(start="2021-09-01", end="2023-12-31", auto_adjust=False)

if raw.empty or "Close" not in raw.columns:
    print("DAY: sem dados")
else:
    close = raw["Close"].copy()
    close.index = pd.to_datetime(close.index).tz_localize(None)
    close = close.dropna().sort_index()

    msft = pd.read_csv(PRICES_DIR / "MSFT.csv", parse_dates=["Date"]).set_index("Date")["MSFT"]
    n_expected = ((msft.index >= "2021-09-01") & (msft.index <= "2023-12-31")).sum()
    cov = len(close) / n_expected * 100
    by_year = close.groupby(close.index.year).count()

    print(f"DAY->CDAY: {len(close)} linhas | {close.index[0].date()} -> {close.index[-1].date()}")
    print(f"Por ano: {by_year.to_dict()}")
    print(f"Cobertura: {cov:.1f}%")

    if cov > 95:
        out = close.rename("CDAY").reset_index()
        out.columns = ["Date", "CDAY"]
        out.to_csv(RECOVERY_DIR / "CDAY.csv", index=False)
        out.to_csv(PRICES_DIR   / "CDAY.csv", index=False)
        print("OK CDAY incorporado de DAY com cobertura completa")
    else:
        print(f"AVISO Cobertura {cov:.1f}% - verificar")


$DAY: possibly delisted; no price data found  (1d 2021-09-01 -> 2023-12-31) (Yahoo error = "No data found, symbol may be delisted")


DAY: sem dados


In [16]:
import requests, time
import pandas as pd
from pathlib import Path
from io import StringIO

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "pt-BR,pt;q=0.9,en;q=0.8",
    "Referer": "https://br.advfn.com/",
}

# Teste rapido: ver o que o ADVFN retorna
url = "https://br.advfn.com/bolsa-de-valores/nyse/CDAY/historico"
r = requests.get(url, headers=HEADERS, timeout=20)
print(f"Status: {r.status_code}")
print(f"Content-Type: {r.headers.get(chr(99)+chr(111)+chr(110)+chr(116)+chr(101)+chr(110)+chr(116)+chr(45)+chr(116)+chr(121)+chr(112)+chr(101))}")
print(f"Tamanho HTML: {len(r.text)} chars")
print()

# Verificar se ha tabela de dados no HTML
has_table = "<table" in r.text.lower()
has_date  = any(x in r.text for x in ["2021", "2022", "2023"])
print(f"Tem <table>: {has_table}")
print(f"Tem datas 2021-2023: {has_date}")

if has_table and has_date:
    # Tentar parsear as tabelas da pagina
    try:
        tables = pd.read_html(StringIO(r.text))
        print(f"Tabelas encontradas: {len(tables)}")
        for i, t in enumerate(tables):
            print(f"  Tabela {i}: {t.shape} | colunas: {list(t.columns)[:5]}")
            if len(t) > 5:
                print(t.head(3).to_string())
    except Exception as e:
        print(f"Erro ao parsear tabelas: {e}")
else:
    print("HTML snippet:")
    print(r.text[:500])


Status: 200
Content-Type: text/html; charset=utf-8
Tamanho HTML: 143473 chars

Tem <table>: True
Tem datas 2021-2023: False
HTML snippet:
<!DOCTYPE HTML PUBLIC "-//W3C//DTD HTML 4.01//EN" "http://www.w3.org/TR/html4/strict.dtd">
<html lang="pt-BR" xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=edge" />
<meta name="format-detection" content="telephone=no" />
<title>Dados Históricos Ceridian HCM Holding Inc - CDAY | Ações New York Stock Exchange | ADVFN Brasil</title>
<!-- Google Tag Manager -->
<script>
    window.data


In [3]:
import requests, time, re
import pandas as pd
from io import StringIO
from pathlib import Path

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "pt-BR,pt;q=0.9",
    "Referer": "https://br.advfn.com/",
}

BASE = "https://br.advfn.com/bolsa-de-valores/nyse/CDAY/historico"

# 1) Ver o que a tabela da pagina padrao contem
r0 = requests.get(BASE, headers=HEADERS, timeout=20)
tables = pd.read_html(StringIO(r0.text))
for i, t in enumerate(tables):
    if len(t) > 3:
        print(f"Tabela {i} ({t.shape}): {list(t.columns)}")
        print(t.head(3).to_string())
        print()

# 2) Tentar URLs com parametros de data (ADVFN aceita mes/ano)
print("--- Testando URLs com parametros ---")
test_urls = [
    BASE + "?Date=202110",
    BASE + "?months=3&startdate=202109",
    BASE + "/2021-10",
    BASE + "?start=2021-09-01&end=2021-12-31",
]
for url in test_urls:
    r = requests.get(url, headers=HEADERS, timeout=15)
    has_dates = any(f"{y}" in r.text for y in ["2021", "2022"])
    try:
        ts = pd.read_html(StringIO(r.text))
        biggest = max(ts, key=len) if ts else None
        n = len(biggest) if biggest is not None else 0
    except:
        n = 0
    print(f"  {url.split(BASE)[1] or "/":<35} status={r.status_code} datas={has_dates} tabela_rows={n}")
    time.sleep(1)


ValueError: No tables found matching pattern '.+'

In [2]:
!pip install html5lib


[notice] A new release of pip is available: 24.3.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import requests, time
import pandas as pd
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")
AV_KEY = "P76D2NTYWAEHW613"

# Alpha Vantage: TIME_SERIES_DAILY com outputsize=full
# campo "4. close" = close split-adjusted (mesmo padrao Yahoo)
url = ("https://www.alphavantage.co/query"
       "?function=TIME_SERIES_DAILY"
       "&symbol=CDAY"
       "&outputsize=full"
       f"&apikey={AV_KEY}")

print("Buscando CDAY no Alpha Vantage...")
r = requests.get(url, timeout=30)
print(f"Status: {r.status_code}")
d = r.json()

if "Time Series (Daily)" not in d:
    print("Erro ou limite atingido:")
    print(d)
else:
    ts = d["Time Series (Daily)"]
    df = pd.DataFrame.from_dict(ts, orient="index")
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()
    df["close"] = pd.to_numeric(df["4. close"], errors="coerce")

    # Filtrar periodo de membership
    cday = df[(df.index >= "2021-09-01") & (df.index <= "2023-12-31")]["close"]
    cday.name = "CDAY"

    msft = pd.read_csv(PRICES_DIR / "MSFT.csv", parse_dates=["Date"]).set_index("Date")["MSFT"]
    n_exp = ((msft.index >= "2021-09-01") & (msft.index <= "2023-12-31")).sum()
    cov = len(cday) / n_exp * 100
    by_year = cday.groupby(cday.index.year).count()

    print(f"CDAY: {len(cday)} linhas | {cday.index[0].date()} -> {cday.index[-1].date()}")
    print(f"Por ano: {by_year.to_dict()}")
    print(f"Cobertura: {cov:.1f}%")
    print(f"Amostra:{cday.head(3).to_string()}")

    if cov > 95:
        out = cday.reset_index()
        out.columns = ["Date", "CDAY"]
        out.to_csv(RECOVERY_DIR / "CDAY.csv", index=False)
        out.to_csv(PRICES_DIR   / "CDAY.csv", index=False)
        print("OK CDAY incorporado via Alpha Vantage")
    else:
        print(f"AVISO cobertura {cov:.1f}%")


Buscando CDAY no Alpha Vantage...
Status: 200
Erro ou limite atingido:
{'Information': 'Thank you for using Alpha Vantage! The outputsize=full parameter value is a premium feature for the TIME_SERIES_DAILY endpoint. You may subscribe to any of the premium plans at https://www.alphavantage.co/premium/ to instantly unlock all premium features'}


In [6]:
import requests, time
import pandas as pd
from pathlib import Path

PRICES_DIR = Path("../data_bases/prices")
AV_KEY = "P76D2NTYWAEHW613"

# compact = ultimos 100 dias de negociacao
# Para tickers inativos, os 100 dias sao os 100 ultimos antes do delisting
# Para tickers ativos, sao os 100 mais recentes

def probe_av(symbol, membership_start, membership_end):
    r = requests.get(
        "https://www.alphavantage.co/query",
        params={"function": "TIME_SERIES_DAILY", "symbol": symbol,
                "outputsize": "compact", "apikey": AV_KEY},
        timeout=20)
    d = r.json()
    if "Time Series (Daily)" not in d:
        msg = d.get("Note", d.get("Information", d.get("Error Message", str(d)[:80])))
        print(f"  [{symbol}] ERRO: {msg[:80]}")
        return
    ts = d["Time Series (Daily)"]
    dates = pd.to_datetime(list(ts.keys())).sort_values()
    in_range = dates[(dates >= membership_start) & (dates <= membership_end)]
    print(f"  [{symbol}] {len(dates)} linhas | {dates[0].date()} -> {dates[-1].date()} "
          f"| no periodo: {len(in_range)}")
    time.sleep(12)  # 5 req/min no free tier

print("=== Alpha Vantage — compact (100 dias) ===")
# CDAY via DAY
probe_av("DAY",  "2021-09-01", "2023-12-31")
# CBS (merged 2019)
probe_av("CBS",  "2016-01-01", "2019-12-05")
# FRC (bankrupt 2023)
probe_av("FRC",  "2018-07-01", "2023-05-01")
# ENDP (bankrupt 2022)
probe_av("ENDP", "2016-01-01", "2022-08-16")
# MNK (bankrupt 2020)
probe_av("MNK",  "2016-01-01", "2020-10-12")


=== Alpha Vantage — compact (100 dias) ===
  [DAY] 100 linhas | 2025-09-11 -> 2026-02-03 | no periodo: 0
  [CBS] ERRO: Invalid API call. Please retry or visit the documentation (https://www.alphavant
  [FRC] ERRO: Invalid API call. Please retry or visit the documentation (https://www.alphavant
  [ENDP] ERRO: Invalid API call. Please retry or visit the documentation (https://www.alphavant
  [MNK] ERRO: Invalid API call. Please retry or visit the documentation (https://www.alphavant


In [8]:
import yfinance as yf
import requests, time, shutil
import pandas as pd
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")
EODHD_KEY    = "69f7ed67d53b32.72220802"

# Tickers ainda ausentes + abordagem
# (orig, successor_yf, eodhd_ticker, start, end)
# successor_yf = None se nao ha sucessor via yfinance
# eodhd_ticker = ticker para tentar no EODHD (geralmente o orig)
targets_final = [
    # --- Reciclados: tentar yfinance com successor ---
    ("FB",    "META",  "FB",    "2016-01-01", "2021-10-28"),
    ("LB",    "BBWI",  "LB",    "2016-01-01", "2021-08-02"),
    ("STI",   "TFC",   "STI",   "2016-01-01", "2019-12-06"),
    ("ARNC",  "HWM",   "ARNC",  "2016-01-01", "2020-04-01"),
    ("DISCA", "WBD",   "DISCA", "2016-01-01", "2022-04-08"),
    ("VIAC",  "PARA",  "VIAC",  "2019-12-05", "2022-02-15"),
    ("CBS",   "PARA",  "CBS",   "2016-01-01", "2019-12-04"),
    ("HCP",   "PEAK",  "HCP",   "2016-01-01", "2019-06-30"),
    ("SE",    "ENB",   "SE",    "2016-01-01", "2017-02-27"),
    ("FTR",   "FYBR",  "FTR",   "2016-01-01", "2020-04-14"),
    # --- Sem successor claro: so EODHD ---
    ("MON",   None,     "MON",   "2016-01-01", "2018-06-07"),
    ("CA",    None,     "CA",    "2016-01-01", "2018-11-05"),
    ("EMC",   None,     "EMC",   "2016-01-01", "2016-09-07"),
    ("DNB",   None,     "DNB",   "2016-01-01", "2019-02-08"),
    ("DO",    None,     "DO",    "2016-01-01", "2020-04-26"),
    ("TE",    None,     "TE",    "2016-01-01", "2016-07-01"),
    ("ENDP",  None,     "ENDP",  "2016-01-01", "2022-08-16"),
    ("MNK",   None,     "MNK",   "2016-01-01", "2020-10-12"),
    ("FRC",   None,     "FRC",   "2018-07-01", "2023-05-01"),
    ("CDAY",  None,     "CDAY",  "2021-09-20", "2023-10-18"),
]


def try_yf_successor(orig, successor, start, end):
    try:
        t = yf.Ticker(successor)
        raw = t.history(start=start, end=end, auto_adjust=False)
        if raw.empty or "Close" not in raw.columns:
            return pd.Series(dtype=float)
        s = raw["Close"].copy()
        s.index = pd.to_datetime(s.index).tz_localize(None)
        return s.dropna().rename(orig)
    except:
        return pd.Series(dtype=float)


def try_eodhd(orig, eodhd_ticker, start, end):
    url = (f"https://eodhd.com/api/eod/{eodhd_ticker}.US"
           f"?from={start}&to={end}&period=d&api_token={EODHD_KEY}&fmt=json")
    try:
        r = requests.get(url, timeout=15)
        data = r.json()
        if not isinstance(data, list) or len(data) < 5:
            return pd.Series(dtype=float)
        df = pd.DataFrame(data)
        df["date"] = pd.to_datetime(df["date"])
        return df.set_index("date")["close"].dropna().rename(orig)
    except:
        return pd.Series(dtype=float)


def save_and_incorporate(orig, s):
    out = s.reset_index()
    out.columns = ["Date", orig]
    out.to_csv(RECOVERY_DIR / f"{orig}.csv", index=False)
    dest = PRICES_DIR / f"{orig}.csv"
    if dest.exists():
        existing = pd.read_csv(dest, parse_dates=["Date"])
        merged = (pd.concat([out, existing])
                  .drop_duplicates("Date").sort_values("Date").reset_index(drop=True))
        merged.to_csv(dest, index=False)
    else:
        out.to_csv(dest, index=False)


results_final = {}
msft = pd.read_csv(PRICES_DIR / "MSFT.csv", parse_dates=["Date"]).set_index("Date")["MSFT"]

print("=" * 60)
print("PARTE G - Tentativa final: yfinance successor + EODHD")
print("=" * 60)

for orig, successor, eodhd_t, start, end in targets_final:
    print(f"[{orig}]  ", end="", flush=True)
    s = pd.Series(dtype=float)

    # 1) yfinance successor
    if successor:
        s = try_yf_successor(orig, successor, start, end)
        if len(s) > 10:
            print(f"yf({successor}) OK  ", end="")

    # 2) EODHD
    if len(s) <= 10:
        s = try_eodhd(orig, eodhd_t, start, end)
        if len(s) > 10:
            print(f"EODHD OK  ", end="")
        time.sleep(1.5)  # EODHD: 20 req/dia, sem rate limit por minuto

    if len(s) > 10:
        n_exp = ((msft.index >= start) & (msft.index <= end)).sum()
        cov = len(s) / n_exp * 100
        save_and_incorporate(orig, s)
        print(f"=> {len(s)} linhas | cobertura {cov:.0f}%")
        results_final[orig] = "ok"
    else:
        print(f"FALHOU (yf+EODHD)")
        results_final[orig] = "fail"

print(f"{'='*60}")
ok   = [t for t, v in results_final.items() if v == "ok"]
fail = [t for t, v in results_final.items() if v == "fail"]
print(f"OK  Recuperados ({len(ok)}): {ok}")
print(f"FAIL Falharam   ({len(fail)}): {fail}")


PARTE G - Tentativa final: yfinance successor + EODHD
[FB]  yf(META) OK  => 1466 linhas | cobertura 100%
[LB]  yf(BBWI) OK  => 1404 linhas | cobertura 100%
[STI]  yf(TFC) OK  => 989 linhas | cobertura 100%
[ARNC]  yf(HWM) OK  => 858 linhas | cobertura 80%
[DISCA]  yf(WBD) OK  => 1578 linhas | cobertura 100%
[VIAC]  

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PARA"}}}
$PARA: possibly delisted; no timezone found


FALHOU (yf+EODHD)
[CBS]  

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: PARA"}}}
$PARA: possibly delisted; no timezone found


FALHOU (yf+EODHD)
[HCP]  

$PEAK: possibly delisted; no timezone found


FALHOU (yf+EODHD)
[SE]  yf(ENB) OK  => 289 linhas | cobertura 100%
[FTR]  

$FYBR: possibly delisted; no timezone found


FALHOU (yf+EODHD)
[MON]  FALHOU (yf+EODHD)
[CA]  FALHOU (yf+EODHD)
[EMC]  FALHOU (yf+EODHD)
[DNB]  FALHOU (yf+EODHD)
[DO]  FALHOU (yf+EODHD)
[TE]  FALHOU (yf+EODHD)
[ENDP]  FALHOU (yf+EODHD)
[MNK]  FALHOU (yf+EODHD)
[FRC]  FALHOU (yf+EODHD)
[CDAY]  FALHOU (yf+EODHD)
OK  Recuperados (6): ['FB', 'LB', 'STI', 'ARNC', 'DISCA', 'SE']
FAIL Falharam   (14): ['VIAC', 'CBS', 'HCP', 'FTR', 'MON', 'CA', 'EMC', 'DNB', 'DO', 'TE', 'ENDP', 'MNK', 'FRC', 'CDAY']


In [9]:
import yfinance as yf
import pandas as pd, shutil
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")

def get_yf(orig, current, start, end):
    t = yf.Ticker(current)
    raw = t.history(start=start, end=end, auto_adjust=False)
    if raw.empty or "Close" not in raw.columns:
        return pd.Series(dtype=float)
    s = raw["Close"].copy()
    s.index = pd.to_datetime(s.index).tz_localize(None)
    return s.dropna().rename(orig)

def save_inc(orig, s):
    out = s.reset_index(); out.columns = ["Date", orig]
    out.to_csv(RECOVERY_DIR / f"{orig}.csv", index=False)
    dest = PRICES_DIR / f"{orig}.csv"
    if dest.exists():
        ex = pd.read_csv(dest, parse_dates=["Date"])
        merged = pd.concat([out, ex]).drop_duplicates("Date").sort_values("Date").reset_index(drop=True)
        merged.to_csv(dest, index=False)
    else:
        out.to_csv(dest, index=False)

# --- HCP via DOC (mesma empresa: HCP->PEAK->DOC) ---
print("[HCP] tentando via DOC (2016-2019)...")
s = get_yf("HCP", "DOC", "2016-01-01", "2019-06-30")
if len(s) > 10:
    save_inc("HCP", s)
    print(f"  OK {len(s)} linhas | {s.index[0].date()} -> {s.index[-1].date()}")
else:
    print("  FALHOU")

# --- VIAC e CBS via PARA (Paramount foi privado dez/2024) ---
# Tentar PARA com yfinance - pode ter dados historicos ate dez/2024
print()
for orig, start, end in [("VIAC", "2019-12-05", "2022-02-15"),
                          ("CBS",  "2016-01-01", "2019-12-04")]:
    print(f"[{orig}] tentando via PARA...")
    s = get_yf(orig, "PARA", start, end)
    if len(s) > 10:
        save_inc(orig, s)
        print(f"  OK {len(s)} linhas | {s.index[0].date()} -> {s.index[-1].date()}")
    else:
        print(f"  FALHOU - PARA delistado dez/2024, dados inacessiveis")


[HCP] tentando via DOC (2016-2019)...


$PARA: possibly delisted; no timezone found
$PARA: possibly delisted; no timezone found


  OK 878 linhas | 2016-01-04 -> 2019-06-28

[VIAC] tentando via PARA...
  FALHOU - PARA delistado dez/2024, dados inacessiveis
[CBS] tentando via PARA...
  FALHOU - PARA delistado dez/2024, dados inacessiveis


In [14]:
import requests, time
import pandas as pd
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")

# Cadastro gratuito em data.nasdaq.com -> Settings -> API Key
NASDAQ_KEY = "4xiLQjDZiwfFHxH2Mv99"   # cole sua key aqui

targets_wiki = [
    ("MON",  "2016-01-01", "2018-06-07"),
    ("CA",   "2016-01-01", "2018-03-27"),
    ("EMC",  "2016-01-01", "2016-09-07"),
    ("FTR",  "2016-01-01", "2016-12-31"),
    ("DO",   "2016-01-01", "2016-06-30"),
    ("TE",   "2016-01-01", "2016-07-01"),
    ("DNB",  "2016-01-01", "2016-12-31"),
    ("MNK",  "2016-01-01", "2017-06-30"),
    ("ENDP", "2016-01-01", "2016-12-31"),
    ("CBS",  "2016-01-01", "2018-03-27"),
]


def fetch_wiki_table(ticker, start, end, key):
    """
    Endpoint correto: datatables/WIKI/PRICES (nao datasets/WIKI/{ticker}).
    Retorna coluna close (preco de fechamento, sem ajuste de dividendos).
    """
    url = "https://data.nasdaq.com/api/v3/datatables/WIKI/PRICES"
    params = {
        "ticker": ticker,
        "date.gte": start,
        "date.lte": end,
        "qopts.columns": "ticker,date,close",
        "api_key": key,
    }
    try:
        r = requests.get(url, params=params, timeout=20)
        if not r.ok:
            print(f"    HTTP {r.status_code}: {r.text[:120]}")
            return pd.Series(dtype=float, name=ticker)
        d = r.json()
        if "datatable" not in d or not d["datatable"]["data"]:
            print(f"    vazio")
            return pd.Series(dtype=float, name=ticker)
        cols = [c["name"] for c in d["datatable"]["columns"]]
        df = pd.DataFrame(d["datatable"]["data"], columns=cols)
        df["date"] = pd.to_datetime(df["date"])
        return df.set_index("date")["close"].rename(ticker).sort_index()
    except Exception as e:
        print(f"    erro: {e}")
        return pd.Series(dtype=float, name=ticker)


def save_and_inc(orig, s):
    out = s.reset_index(); out.columns = ["Date", orig]
    out.to_csv(RECOVERY_DIR / f"{orig}.csv", index=False)
    dest = PRICES_DIR / f"{orig}.csv"
    if dest.exists():
        ex = pd.read_csv(dest, parse_dates=["Date"])
        merged = (pd.concat([out, ex])
                  .drop_duplicates("Date").sort_values("Date").reset_index(drop=True))
        merged.to_csv(dest, index=False)
        return len(merged)
    else:
        out.to_csv(dest, index=False)
        return len(out)


# --- Teste rapido antes de rodar tudo ---
if not NASDAQ_KEY:
    print("NASDAQ_KEY nao definida.")
    print("1. Cadastre-se em https://data.nasdaq.com (gratis, sem cartao)")
    print("2. Vá em Account Settings -> Your API Key")
    print("3. Cole a key acima e reexecute.")
else:
    print("Testando endpoint com AAPL...")
    test = fetch_wiki_table("AAPL", "2016-01-01", "2016-01-10", NASDAQ_KEY)
    if len(test) > 0:
        print(f"OK endpoint funciona: {test.to_dict()}")
        print()

        msft = pd.read_csv(PRICES_DIR / "MSFT.csv", parse_dates=["Date"]).set_index("Date")["MSFT"]
        results_wiki = {}

        print("=" * 55)
        print("PARTE H - NASDAQ Data Link WIKI/PRICES")
        print("=" * 55)

        for ticker, start, end in targets_wiki:
            print(f"[{ticker}]  ", end="", flush=True)
            s = fetch_wiki_table(ticker, start, end, NASDAQ_KEY)
            if len(s) > 10:
                n_exp = ((msft.index >= start) & (msft.index <= end)).sum()
                cov = len(s) / n_exp * 100 if n_exp > 0 else 0
                save_and_inc(ticker, s)
                print(f"OK  {len(s)} linhas | cov {cov:.0f}%")
                results_wiki[ticker] = "ok"
            else:
                print(f"FALHOU")
                results_wiki[ticker] = "fail"
            time.sleep(0.3)

        ok   = [t for t, v in results_wiki.items() if v == "ok"]
        fail = [t for t, v in results_wiki.items() if v == "fail"]
        print(f"OK   ({len(ok)}): {ok}")
        print(f"FAIL ({len(fail)}): {fail}")
    else:
        print("Endpoint AAPL falhou - WIKI/PRICES pode requerer plano pago.")
        print("Tente: pip install nasdaq-data-link; import nasdaqdatalink; nasdaqdatalink.get_table(...)")


Testando endpoint com AAPL...
OK endpoint funciona: {Timestamp('2016-01-04 00:00:00'): 105.35, Timestamp('2016-01-05 00:00:00'): 102.71, Timestamp('2016-01-06 00:00:00'): 100.7, Timestamp('2016-01-07 00:00:00'): 96.45, Timestamp('2016-01-08 00:00:00'): 96.96}

PARTE H - NASDAQ Data Link WIKI/PRICES
[MON]  OK  561 linhas | cov 92%
[CA]  OK  561 linhas | cov 100%
[EMC]  OK  171 linhas | cov 99%
[FTR]  OK  252 linhas | cov 100%
[DO]  OK  125 linhas | cov 100%
[TE]  OK  125 linhas | cov 99%
[DNB]  OK  252 linhas | cov 100%
[MNK]  OK  377 linhas | cov 100%
[ENDP]  OK  252 linhas | cov 100%
[CBS]  OK  561 linhas | cov 100%
OK   (10): ['MON', 'CA', 'EMC', 'FTR', 'DO', 'TE', 'DNB', 'MNK', 'ENDP', 'CBS']
FAIL (0): []


In [20]:
import yfinance as yf
import requests, time
import pandas as pd
from pathlib import Path

PRICES_DIR   = Path("../data_bases/prices")
RECOVERY_DIR = Path("../data_bases/external/macrotrends_recovery")
NASDAQ_KEY   = "4xiLQjDZiwfFHxH2Mv99"   # mesma key da Parte H

FINANCIALDATA_KEY = "ff6ca5dffb9951ff93400949958695de"


def get_yf(orig, current, start, end):
    t = yf.Ticker(current)
    raw = t.history(start=start, end=end, auto_adjust=False)
    if raw.empty or "Close" not in raw.columns:
        return pd.Series(dtype=float)
    s = raw["Close"].copy()
    s.index = pd.to_datetime(s.index).tz_localize(None)
    return s.dropna().sort_index().rename(orig)


def get_wiki(ticker, start, end):
    r = requests.get("https://data.nasdaq.com/api/v3/datatables/WIKI/PRICES",
        params={"ticker": ticker, "date.gte": start, "date.lte": end,
                "qopts.columns": "ticker,date,close", "api_key": NASDAQ_KEY}, timeout=20)
    if not r.ok: return pd.Series(dtype=float)
    d = r.json()
    if "datatable" not in d or not d["datatable"]["data"]: return pd.Series(dtype=float)
    cols = [c["name"] for c in d["datatable"]["columns"]]
    df = pd.DataFrame(d["datatable"]["data"], columns=cols)
    df["date"] = pd.to_datetime(df["date"])
    return df.set_index("date")["close"].sort_index().rename(ticker)


def get_financialdata_paginated(ticker, start, end):
    records, offset, start_dt = [], 0, pd.to_datetime(start)
    while True:
        r = requests.get("https://financialdata.net/api/v1/stock-prices",
            params={"identifier": ticker, "key": FINANCIALDATA_KEY, "format": "json", "offset": offset},
            timeout=15)
        if not r.ok: break
        page = r.json()
        if not isinstance(page, list) or not page: break
        records.extend(page)
        if pd.to_datetime(page[-1]["date"]) <= start_dt: break
        offset += 300
        time.sleep(0.4)
    if not records: return pd.Series(dtype=float)
    df = pd.DataFrame(records)
    df["date"] = pd.to_datetime(df["date"])
    df = df[(df["date"] >= start) & (df["date"] <= end)]
    return df.sort_values("date").set_index("date")["close"].rename(ticker)


def merge_with_existing(orig, s_new):
    """Adiciona dados novos ao arquivo existente (sem sobrescrever o que ja tem)."""
    dest = PRICES_DIR / f"{orig}.csv"
    s_new = s_new.dropna()
    if len(s_new) == 0: return 0
    new_df = s_new.reset_index(); new_df.columns = ["Date", orig]
    if dest.exists():
        existing = pd.read_csv(dest, parse_dates=["Date"])
        merged = (pd.concat([new_df, existing])
                  .drop_duplicates("Date").sort_values("Date").reset_index(drop=True))
        merged.to_csv(dest, index=False)
        return len(merged)
    else:
        new_df.to_csv(dest, index=False)
        return len(new_df)


msft = pd.read_csv(PRICES_DIR / "MSFT.csv", parse_dates=["Date"]).set_index("Date")["MSFT"]

print("=" * 60)
print("PARTE I — Tentativas para tickers parciais")
print("=" * 60)

# ── IR: falta 2016/S1-2016/S2 ──────────────────────────────
print("[IR] falta 2016-01-01 a 2017-05-11")
# 1) TT (Trane Technologies = sucessor legal do old Ingersoll-Rand)
s = get_yf("IR", "TT", "2016-01-01", "2017-05-11")
if len(s) > 10:
    n = merge_with_existing("IR", s)
    print(f"  yf(TT) OK: {len(s)} linhas novas | total {n} em prices/IR.csv")
else:
    print(f"  yf(TT) falhou ({len(s)} linhas)")
    if NASDAQ_KEY:
        s = get_wiki("IR", "2016-01-01", "2017-05-11")
        if len(s) > 10:
            n = merge_with_existing("IR", s)
            print(f"  WIKI OK: {len(s)} linhas novas | total {n} em prices/IR.csv")
        else:
            print(f"  WIKI falhou ({len(s)} linhas)")
    else:
        print("  (NASDAQ_KEY nao definida — pular WIKI)")

# ── FOX/FOXA: 21CF 2016-2019 ───────────────────────────────
print()
for ticker in ["FOX", "FOXA"]:
    print(f"[{ticker}] falta 21CF 2016/S1 → 2018/S2")
    s_total = pd.Series(dtype=float)
    # WIKI cobre 21CF até mar/2018 (eles usavam esses tickers ate mar/2019)
    if NASDAQ_KEY:
        s = get_wiki(ticker, "2016-01-01", "2018-03-27")
        if len(s) > 10:
            n = merge_with_existing(ticker, s)
            print(f"  WIKI (2016-2018): {len(s)} linhas | total {n} em prices/{ticker}.csv")
            s_total = s
        else:
            print(f"  WIKI falhou ({len(s)} linhas)")
        time.sleep(0.5)
    else:
        print("  (NASDAQ_KEY nao definida — pular WIKI)")
    # Para 2018/S1 restante (abr-jun 2018) e 2018/S2: sem fonte gratuita conhecida
    n_exp_total = ((msft.index >= "2016-01-01") & (msft.index <= "2018-12-31")).sum()
    if len(s_total) > 0:
        cov = len(s_total) / n_exp_total * 100
        print(f"  Cobertura do periodo 21CF (2016-2018): {cov:.0f}%")
        print(f"  Nota: 2018/S1 (abr-jun) e 2018/S2 ficam sem fonte gratuita")

# ── GPS: gap 2020-2021 ──────────────────────────────────────
print()
print("[GPS] gap 2020/S1 → 2021/S2")
# Financial Data API paginado (GPS retornou 0 antes — tentar novamente)
s = get_financialdata_paginated("GPS", "2020-01-01", "2021-12-31")
if len(s) > 10:
    n = merge_with_existing("GPS", s)
    print(f"  Financial Data OK: {len(s)} linhas | total {n} em prices/GPS.csv")
else:
    print(f"  Financial Data falhou ({len(s)} linhas)")
    print("  Sugestão: baixar GPS manualmente em finance.yahoo.com/quote/GPS/history")
    print(" Time Period: 01/01/2020 → 31/12/2021 → Download")
    print('Depois usar process_yahoo_web("GPS", "2020-1-1", "2021-12-31")')


PARTE I — Tentativas para tickers parciais
[IR] falta 2016-01-01 a 2017-05-11
  yf(TT) OK: 341 linhas novas | total 2512 em prices/IR.csv

[FOX] falta 21CF 2016/S1 → 2018/S2
  WIKI (2016-2018): 561 linhas | total 2272 em prices/FOX.csv
  Cobertura do periodo 21CF (2016-2018): 74%
  Nota: 2018/S1 (abr-jun) e 2018/S2 ficam sem fonte gratuita
[FOXA] falta 21CF 2016/S1 → 2018/S2


ValueError: Missing column provided to 'parse_dates': 'Date'

In [24]:
import requests, time
import pandas as pd
from pathlib import Path

PRICES_DIR        = Path("../data_bases/prices")
NASDAQ_KEY        = "ff6ca5dffb9951ff93400949958695de"  # mesma key da Parte H
FINANCIALDATA_KEY = "ff6ca5dffb9951ff93400949958695de"


def get_wiki(ticker, start, end):
    r = requests.get("https://data.nasdaq.com/api/v3/datatables/WIKI/PRICES",
        params={"ticker": ticker, "date.gte": start, "date.lte": end,
                "qopts.columns": "ticker,date,close", "api_key": NASDAQ_KEY}, timeout=20)
    if not r.ok: return pd.Series(dtype=float)
    d = r.json()
    if "datatable" not in d or not d["datatable"]["data"]: return pd.Series(dtype=float)
    cols = [c["name"] for c in d["datatable"]["columns"]]
    df = pd.DataFrame(d["datatable"]["data"], columns=cols)
    df["date"] = pd.to_datetime(df["date"])
    return df.set_index("date")["close"].sort_index().rename(ticker)


def merge_robust(orig, s_new):
    """Merge tolerante a diferentes formatos de CSV existentes."""
    dest = PRICES_DIR / f"{orig}.csv"
    s_new = s_new.dropna()
    if len(s_new) == 0: return 0
    new_df = s_new.reset_index()
    new_df.columns = ["Date", orig]
    if dest.exists():
        existing = pd.read_csv(dest)
        # Detectar coluna de data (pode ser Date, date, Unnamed: 0, etc.)
        date_col = next((c for c in existing.columns
                         if "date" in c.lower() or "unnamed" in c.lower()),
                        existing.columns[0])
        existing = existing.rename(columns={date_col: "Date"})
        existing["Date"] = pd.to_datetime(existing["Date"])
        price_col = [c for c in existing.columns if c != "Date"][0]
        existing = existing.rename(columns={price_col: orig})
        merged = (pd.concat([new_df, existing])
                  .drop_duplicates("Date").sort_values("Date").reset_index(drop=True))
        merged.to_csv(dest, index=False)
        return len(merged)
    else:
        new_df.to_csv(dest, index=False)
        return len(new_df)


# ── FOXA: WIKI 2016-2018 (21CF) ─────────────────────────────
print("[FOXA] WIKI 2016-mar/2018 (21CF)...")
if NASDAQ_KEY:
    s = get_wiki("FOXA", "2016-01-01", "2018-03-27")
    if len(s) > 10:
        n = merge_robust("FOXA", s)
        print(f"  OK: {len(s)} linhas | total {n} em prices/FOXA.csv")
    else:
        print(f"  WIKI falhou ({len(s)} linhas)")
else:
    print("  NASDAQ_KEY nao definida")

# ── GPS: Financial Data API 2020-2021 ───────────────────────
print()
print("[GPS] Financial Data API 2020-2021...")
records, offset, start_dt = [], 0, pd.to_datetime("2020-01-01")
while True:
    r = requests.get("https://financialdata.net/api/v1/stock-prices",
        params={"identifier": "GPS", "key": FINANCIALDATA_KEY,
                "format": "json", "offset": offset}, timeout=15)
    if not r.ok: break
    page = r.json()
    if not isinstance(page, list) or not page: break
    records.extend(page)
    oldest = pd.to_datetime(page[-1]["date"])
    if oldest <= start_dt: break
    offset += 300
    time.sleep(0.4)

if records:
    df_gps = pd.DataFrame(records)
    df_gps["date"] = pd.to_datetime(df_gps["date"])
    df_gps = df_gps[(df_gps["date"] >= "2020-01-01") & (df_gps["date"] <= "2021-12-31")]
    s_gps = df_gps.set_index("date")["close"].sort_index().rename("GPS") if len(df_gps) > 0 else pd.Series(dtype=float)
    if len(s_gps) > 10:
        n = merge_robust("GPS", s_gps)
        print(f"  Financial Data OK: {len(s_gps)} linhas | total {n} em prices/GPS.csv")
    else:
        print(f"  Financial Data sem dados 2020-2021 ({len(s_gps)} linhas)")
        print("  => baixar manualmente em finance.yahoo.com/quote/GPS/history")
        print("     Time Period: 01/01/2020 -> 31/12/2021 -> Download")
        print('     Depois: process_yahoo_web("GPS", "2020-1-1", "2021-12-31")')
else:
    print("  sem resposta da API")
    print("  => baixar manualmente em finance.yahoo.com/quote/GPS/history")


[FOXA] WIKI 2016-mar/2018 (21CF)...
  WIKI falhou (0 linhas)

[GPS] Financial Data API 2020-2021...
  sem resposta da API
  => baixar manualmente em finance.yahoo.com/quote/GPS/history
